In [1]:
# =============================================================================
# POLK COUNTY IOWA — MULTI-TEMPORAL CORN CLASSIFICATION & CDL VALIDATION
# Complete Pipeline — Steps 0 through 8
# =============================================================================
# PREREQUISITES:
#   - NASA Earthdata account: https://urs.earthdata.nasa.gov/
#   - Set environment variables:
#       export EARTHDATA_USER="your_username"
#       export EARTHDATA_PASS="your_password"
#   - ~10 GB disk space
#   - SageMaker ml.t3.medium or equivalent (CPU is fine, GPU is faster)
#
# USAGE:
#   Run each cell in order. Steps 2 and 3 cache downloads automatically
#   so re-runs are fast. The full pipeline takes ~15-20 minutes on CPU.
# =============================================================================
 
 
# =============================================================================
# STEP 0 — INSTALL DEPENDENCIES
# =============================================================================
import subprocess, sys
 
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
 
pip_install("git+https://github.com/IBM/terratorch.git")
pip_install(
    "rasterio", "geopandas", "shapely", "requests",
    "matplotlib", "scikit-learn", "pystac-client",
    "odc-stac", "planetary-computer", "pyproj",
    "tqdm", "huggingface_hub", "torchvision==0.26.0",
)
 
print("✅ Dependencies installed")
 
 


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
jupyter-ai 2.31.7 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is not installed.
skops 0.13.0 requires prettytable>=3.9, which is not installed.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you have numpy 2.4.4 which is incompatible.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.4 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which i

✅ Dependencies installed


In [2]:
# =============================================================================
# STEP 1 — IMPORTS & GLOBAL CONFIGURATION
# =============================================================================
 
import os
import json
import math
import warnings
import numpy as np
import torch
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
 
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from shapely.geometry import box, mapping
from rasterio.merge import merge
from rasterio.mask import mask as rio_mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import from_bounds
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
 
import albumentations
import lightning.pytorch as pl
 
warnings.filterwarnings("ignore")
 
# ---------------------------------------------------------------------------
# NASA Earthdata credentials
# ---------------------------------------------------------------------------
EARTHDATA_USER = os.environ.get("EARTHDATA_USER", "YOUR_EARTHDATA_USERNAME")
EARTHDATA_PASS = os.environ.get("EARTHDATA_PASS", "YOUR_EARTHDATA_PASSWORD")
 
# ---------------------------------------------------------------------------
# County boundary — loads real polygon from Census TIGER shapefile
# ---------------------------------------------------------------------------
def get_county_boundary(state_fips: str, county_name: str) -> gpd.GeoDataFrame:
    url = (
        "https://www2.census.gov/geo/tiger/TIGER2022/COUNTY/"
        "tl_2022_us_county.zip"
    )
    print("   Downloading Census TIGER county boundaries...", end=" ")
    counties = gpd.read_file(url)
    result = counties[
        (counties["STATEFP"] == state_fips) &
        (counties["NAME"]    == county_name)
    ].to_crs("EPSG:4326")
    if result.empty:
        raise ValueError(f"County '{county_name}' not found for state FIPS '{state_fips}'.")
    print("✓")
    return result
 
print("Loading Polk County boundary...")
polk_gdf  = get_county_boundary(state_fips="19", county_name="Polk")
POLK_GEOM = polk_gdf.geometry.iloc[0]
_bounds   = POLK_GEOM.bounds
POLK_BBOX = {
    "min_lon": _bounds[0], "min_lat": _bounds[1],
    "max_lon": _bounds[2], "max_lat": _bounds[3],
}
 
# ---------------------------------------------------------------------------
# HLS acquisition windows
# ---------------------------------------------------------------------------
HLS_TIMESTEPS = [
    "2022-04-15/2022-04-30",   # T1 — early spring / planting
    "2022-06-15/2022-06-30",   # T2 — canopy development
    "2022-08-01/2022-08-15",   # T3 — peak biomass
]
HLS_YEAR      = 2022
HLS_MAX_ITEMS = 50
HLS_MAX_CLOUD = 20
 
# ---------------------------------------------------------------------------
# Band mapping
# ---------------------------------------------------------------------------
HLS_BANDS = {
    "BLUE":       "B02",
    "GREEN":      "B03",
    "RED":        "B04",
    "NIR_NARROW": "B8A",
    "SWIR_1":     "B11",
    "SWIR_2":     "B12",
}
BAND_ORDER = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]
 
# ---------------------------------------------------------------------------
# Prithvi normalization stats (from checkpoint meta — exact training values)
# ---------------------------------------------------------------------------
PRITHVI_MEANS = np.array([
    494.905781, 815.239594, 924.335066, 2968.881459, 2634.621962, 1739.579917,
    494.905781, 815.239594, 924.335066, 2968.881459, 2634.621962, 1739.579917,
    494.905781, 815.239594, 924.335066, 2968.881459, 2634.621962, 1739.579917,
], dtype=np.float32)
 
PRITHVI_STDS = np.array([
    284.925432, 357.84876, 575.566823, 896.601013, 951.900334, 921.407808,
    284.925432, 357.84876, 575.566823, 896.601013, 951.900334, 921.407808,
    284.925432, 357.84876, 575.566823, 896.601013, 951.900334, 921.407808,
], dtype=np.float32)
 
# ---------------------------------------------------------------------------
# Model config
# ---------------------------------------------------------------------------
CHIP_SIZE    = 224
CHIP_STRIDE  = 224
NUM_FRAMES   = 3
NUM_CLASSES  = 13
HEAD_DROPOUT = 0.1
 
CLASS_NAMES = [
    "Corn", "Soybean", "Spring Wheat", "Winter Wheat", "Rice",
    "Sorghum", "Other Small Grain", "Rye", "Oats", "Cotton",
    "Winter Canola", "Winter Rapeseed", "Barley",
]
CORN_CLASS_IDX = 0
CDL_CORN_VALUE = 1
 
CLASS_WEIGHTS = [
    0.386375, 0.661126, 0.548184, 0.640482, 0.876862, 0.925186, 3.249462,
    1.542289, 2.175141, 2.272419, 3.062762, 3.626097, 1.198702
]
 
# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------
OUT_DIR   = Path("./polk_corn_validation")
HLS_DIR   = OUT_DIR / "hls_tiles"
CDL_DIR   = OUT_DIR / "cdl"
CHIPS_DIR = OUT_DIR / "chips"
PRED_DIR  = OUT_DIR / "predictions"
PLOTS_DIR = OUT_DIR / "plots"
 
for d in [HLS_DIR, CDL_DIR, CHIPS_DIR, PRED_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
 
pl.seed_everything(42)
 
print("✅ Configuration complete.")
print(f"   Target   : Polk County, Iowa")
print(f"   Bbox     : {POLK_BBOX}")
print(f"   Timesteps: {HLS_TIMESTEPS}")
print(f"   Output   : {OUT_DIR.resolve()}")
 
 


Loading Polk County boundary...

Seed set to 42


✓
✅ Configuration complete.
   Target   : Polk County, Iowa
   Bbox     : {'min_lon': -93.815784, 'min_lat': 41.48177, 'max_lon': -93.328407, 'max_lat': 41.863469}
   Timesteps: ['2022-04-15/2022-04-30', '2022-06-15/2022-06-30', '2022-08-01/2022-08-15']
   Output   : /home/sagemaker-user/polk_corn_validation


In [3]:
# =============================================================================
# STEP 2 — DOWNLOAD HLS S30 IMAGERY FROM NASA EARTHDATA
# =============================================================================

from rasterio.merge import merge as rio_merge
from rasterio.warp import transform_geom
from shapely.geometry import mapping


def build_earthdata_session() -> requests.Session:
    session = requests.Session()
    session.auth = (EARTHDATA_USER, EARTHDATA_PASS)
    adapter = requests.adapters.HTTPAdapter(max_retries=3)
    session.mount("https://", adapter)
    return session


def search_hls_granules(bbox: dict, date_range: str, max_cloud: float = HLS_MAX_CLOUD) -> list:
    import pystac_client
    catalog   = pystac_client.Client.open("https://cmr.earthdata.nasa.gov/stac/LPCLOUD")
    bbox_list = [bbox["min_lon"], bbox["min_lat"], bbox["max_lon"], bbox["max_lat"]]
    search = catalog.search(
        collections=["HLSS30.v2.0"],
        bbox=bbox_list,
        datetime=date_range,
        query={"eo:cloud_cover": {"lte": max_cloud}},
        max_items=HLS_MAX_ITEMS,
    )
    items = list(search.items())
    print(f"   Found {len(items)} granule(s) for {date_range}")
    items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
    return items


def download_band(asset_href: str, dest_path: Path, session: requests.Session) -> bool:
    if dest_path.exists():
        return True
    try:
        resp = session.get(asset_href, stream=True, timeout=120)
        resp.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        return True
    except Exception as e:
        print(f"   ⚠️  Failed: {e}")
        return False


def download_all_granules_for_timestep(
    timestep_idx: int, date_range: str, bbox: dict, session: requests.Session,
) -> dict | None:
    print(f"\n📡 Timestep {timestep_idx + 1} — {date_range}")
    items = search_hls_granules(bbox, date_range)
    if not items:
        return None

    best_per_tile = {}
    for item in items:
        parts   = item.id.split(".")
        tile_id = next((p for p in parts if p.startswith("T") and len(p) == 6), item.id)
        if tile_id not in best_per_tile:
            best_per_tile[tile_id] = item

    # FIX 1 — cap to 2 tiles max to avoid large mosaics
    best_per_tile = dict(list(best_per_tile.items())[:2])
    print(f"   MGRS tiles: {list(best_per_tile.keys())}")
    band_paths = {b: [] for b in HLS_BANDS}

    for tile_id, item in best_per_tile.items():
        cloud = item.properties.get("eo:cloud_cover", "?")
        print(f"   Granule: {item.id}  (cloud: {cloud}%)")
        for band_name, hls_band_id in HLS_BANDS.items():
            asset_key = next((k for k in item.assets if hls_band_id in k.upper()), None)
            # FIX 2 — skip missing asset instead of killing the whole timestep
            if asset_key is None:
                print(f"      {band_name}... SKIPPED (asset not found)")
                continue
            href = item.assets[asset_key].href
            dest = HLS_DIR / f"t{timestep_idx}_{item.id}_{hls_band_id}.tif"
            print(f"      {band_name}...", end=" ")
            ok = download_band(href, dest, session)
            print("✓" if ok else "FAILED")
            if not ok:
                print(f"      ⚠️  Skipping {band_name} for {tile_id}")
                continue
            band_paths[band_name].append(dest)

    # FIX 3 — don't return a dict with empty band lists; fail clearly
    missing = [b for b in BAND_ORDER if not band_paths[b]]
    if missing:
        print(f"   ⚠️  Missing bands after download: {missing}")
        return None

    return band_paths


def mosaic_and_clip_bands(band_paths: dict, county_geom, out_path: Path) -> Path:
    import tempfile
    clipped_bands = []
    meta = None

    for band_name in BAND_ORDER:
        paths = band_paths[band_name]
        if len(paths) == 1:
            with rasterio.open(paths[0]) as src_ctx:
                mosaic_arr       = src_ctx.read()
                mosaic_transform = src_ctx.transform
                mosaic_crs       = src_ctx.crs
        else:
            srcs = [rasterio.open(p) for p in paths]
            mosaic_arr, mosaic_transform = rio_merge(srcs)
            mosaic_crs = srcs[0].crs
            for s in srcs:
                s.close()

        geom_native = transform_geom("EPSG:4326", mosaic_crs, mapping(county_geom))

        with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as tmp:
            tmp_path = Path(tmp.name)

        with rasterio.open(tmp_path, "w", driver="GTiff",
                           dtype=mosaic_arr.dtype, count=mosaic_arr.shape[0],
                           height=mosaic_arr.shape[1], width=mosaic_arr.shape[2],
                           crs=mosaic_crs, transform=mosaic_transform) as tmp_ds:
            tmp_ds.write(mosaic_arr)

        with rasterio.open(tmp_path) as tmp_ds:
            arr, clip_transform = rio_mask(tmp_ds, [geom_native], crop=True)
            if meta is None:
                meta = tmp_ds.meta.copy()
                meta.update({
                    "count":     len(BAND_ORDER),
                    "height":    arr.shape[1],
                    "width":     arr.shape[2],
                    "transform": clip_transform,
                    "nodata":    0,
                })
        tmp_path.unlink(missing_ok=True)
        clipped_bands.append(arr[0])

    stacked = np.stack(clipped_bands, axis=0)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(stacked)
    return out_path


# RUN
session        = build_earthdata_session()
timestep_stacks = []

print("\n" + "=" * 60)
print(" DOWNLOADING HLS S30 IMAGERY — 3 TIMESTEPS")
print("=" * 60)

for t_idx, date_range in enumerate(HLS_TIMESTEPS):
    band_paths = download_all_granules_for_timestep(t_idx, date_range, POLK_BBOX, session)
    if band_paths is None:
        raise RuntimeError(f"Failed for timestep {t_idx + 1}. Check credentials / dates.")
    stack_path = HLS_DIR / f"polk_hls_t{t_idx}_stacked.tif"
    print(f"\n   Mosaicking & clipping → {stack_path.name} ...", end=" ")
    mosaic_and_clip_bands(band_paths, POLK_GEOM, stack_path)
    print("✓")
    with rasterio.open(stack_path) as src:
        print(f"   Shape: {src.count}b × {src.height} × {src.width}px  CRS: {src.crs}")
    timestep_stacks.append(stack_path)

print(f"\n✅ All {len(timestep_stacks)} timestep stacks saved.")


 DOWNLOADING HLS S30 IMAGERY — 3 TIMESTEPS

📡 Timestep 1 — 2022-04-15/2022-04-30
   Found 5 granule(s) for 2022-04-15/2022-04-30
   MGRS tiles: ['T15TVF', 'T15TVG']
   Granule: HLS.S30.T15TVF.2022106T165839.v2.0  (cloud: 0%)
      BLUE... ✓
      GREEN... ✓
      RED... ✓
      NIR_NARROW... ✓
      SWIR_1... ✓
      SWIR_2... ✓
   Granule: HLS.S30.T15TVG.2022106T165839.v2.0  (cloud: 1%)
      BLUE... ✓
      GREEN... ✓
      RED... ✓
      NIR_NARROW... ✓
      SWIR_1... ✓
      SWIR_2... ✓

   Mosaicking & clipping → polk_hls_t0_stacked.tif ... ✓
   Shape: 6b × 1422 × 1350px  CRS: PROJCS["UTM Zone 15, Northern Hemisphere",GEOGCS["Unknown datum based upon the WGS 84 ellipsoid",DATUM["Not specified (based on WGS 84 spheroid)",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-93],PARAMETER["s

In [4]:
# =============================================================================
# STEP 3 — USDA NASS CDL DOWNLOAD + CLIP TO COUNTY
# =============================================================================

import time
import sys


def live_print(*args):
    print(*args, flush=True)


def _stream_download(url: str, out_path: Path, min_bytes: int = 1_000_000) -> Path | None:
    if out_path.exists():
        live_print(f"   ✔ Cached: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
        return out_path
    live_print(f"\n   📥 {out_path.name}")
    try:
        r = requests.get(url, stream=True, timeout=(20, 30))
    except Exception as e:
        live_print(f"   ❌ {e}")
        return None
    if r.status_code != 200:
        live_print(f"   ❌ HTTP {r.status_code}")
        return None
    total      = int(r.headers.get("content-length", 0))
    downloaded = 0
    last_chunk = time.time()
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if time.time() - last_chunk > 60:
                live_print("\n   ❌ Stalled — aborting download")
                out_path.unlink(missing_ok=True)
                return None
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                last_chunk  = time.time()
                sys.stdout.write(f"\r   ⬇ {downloaded/1e6:.1f} MB")
                sys.stdout.flush()
    live_print(f"\n   ✔ {downloaded/1e6:.1f} MB downloaded")
    if downloaded < min_bytes:
        live_print(f"   ❌ File too small ({downloaded} bytes) — discarding")
        out_path.unlink(missing_ok=True)
        return None
    return out_path


def clip_cdl_to_county(state_cdl_path: Path, county_geom, out_path: Path) -> Path:
    from rasterio.warp import transform_geom
    with rasterio.open(state_cdl_path) as src:
        geom_native = transform_geom("EPSG:4326", src.crs, mapping(county_geom))
        arr, clip_transform = rio_mask(src, [geom_native], crop=True, nodata=0)
        meta = src.meta.copy()
        meta.update({
            "height":    arr.shape[1],
            "width":     arr.shape[2],
            "transform": clip_transform,
            "nodata":    0,
        })
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr)
    live_print(f"   ✔ Clipped CDL: {out_path.name} ({out_path.stat().st_size/1e6:.2f} MB)")
    return out_path


def download_and_clip_cdl(year: int, state_fips: str, county_geom, bbox: dict) -> Path:
    state_raw_path   = CDL_DIR / f"cdl_{year}_state_{state_fips}_raw.tif"
    county_clip_path = CDL_DIR / f"cdl_{year}_county_clipped.tif"
    if county_clip_path.exists():
        live_print(f"   ✔ Cached: {county_clip_path}")
        return county_clip_path
    url    = f"https://nassgeodata.gmu.edu/nass_data_cache/byfips/CDL_{year}_{state_fips}.tif"
    result = _stream_download(url, state_raw_path, min_bytes=50_000_000)
    if result is None:
        raise RuntimeError("CDL download failed — check URL or network.")
    with rasterio.open(state_raw_path) as src:
        live_print(f"   CRS: {src.crs}  Shape: {src.width}×{src.height}")
    live_print("   Clipping to county boundary...")
    clip_cdl_to_county(state_raw_path, county_geom, county_clip_path)
    return county_clip_path


# RUN
live_print("\n" + "=" * 60)
live_print(" STEP 3 — CDL DOWNLOAD + COUNTY CLIP")
live_print("=" * 60)
cdl_path = download_and_clip_cdl(HLS_YEAR, "19", POLK_GEOM, POLK_BBOX)
live_print(f"\n✅ County CDL ready: {cdl_path}")


 STEP 3 — CDL DOWNLOAD + COUNTY CLIP
   ✔ Cached: polk_corn_validation/cdl/cdl_2022_county_clipped.tif

✅ County CDL ready: polk_corn_validation/cdl/cdl_2022_county_clipped.tif


In [6]:
# =============================================================================
# STEP 4 — CHIP IMAGERY INTO 224×224 TILES
# =============================================================================

from rasterio.enums import Resampling as RIOResampling
from rasterio.warp import reproject as rio_reproject


def normalize_chip(chip: np.ndarray, t_idx: int) -> np.ndarray:
    start = t_idx * len(BAND_ORDER)
    means = PRITHVI_MEANS[start:start + len(BAND_ORDER)].reshape(-1, 1, 1)
    stds  = PRITHVI_STDS [start:start + len(BAND_ORDER)].reshape(-1, 1, 1)
    nodata_mask = np.all(chip == 0, axis=0)
    chip_f = (chip.astype(np.float32) - means) / (stds + 1e-6)
    chip_f[:, nodata_mask] = 0.0
    return chip_f


def align_stack_to_reference(src_path, ref_transform, ref_crs, ref_h, ref_w, n_bands):
    aligned = np.zeros((n_bands, ref_h, ref_w), dtype=np.float32)
    with rasterio.open(src_path) as src:
        for b in range(1, n_bands + 1):
            rio_reproject(
                source=rasterio.band(src, b), destination=aligned[b - 1],
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref_transform, dst_crs=ref_crs,
                resampling=RIOResampling.nearest,
            )
    return aligned


def extract_chips(stack_paths, chip_size=CHIP_SIZE, stride=CHIP_STRIDE):
    if len(stack_paths) != NUM_FRAMES:
        raise ValueError(f"Expected {NUM_FRAMES} stacks, got {len(stack_paths)}")

    arrays   = []
    meta_ref = None
    ref_h = ref_w = ref_transform = ref_crs = None

    for t_idx, sp in enumerate(stack_paths):
        if t_idx == 0:
            with rasterio.open(sp) as src:
                arr = src.read().astype(np.float32)
                ref_h, ref_w   = arr.shape[1], arr.shape[2]
                ref_transform  = src.transform
                ref_crs        = src.crs
                meta_ref       = {"transform": ref_transform, "crs": ref_crs}
        else:
            with rasterio.open(sp) as src:
                needs_align = (src.height != ref_h or src.width != ref_w or src.crs != ref_crs)
            if needs_align:
                print(f"   ⚠️  T{t_idx} shape differs — reprojecting")
                arr = align_stack_to_reference(sp, ref_transform, ref_crs, ref_h, ref_w, len(BAND_ORDER))
            else:
                with rasterio.open(sp) as src:
                    arr = src.read().astype(np.float32)
        arrays.append(normalize_chip(arr, t_idx))

    with rasterio.open(stack_paths[0]) as src:
        scene_nodata = np.all(src.read().astype(np.float32) == 0, axis=0)

    scene = np.stack(arrays, axis=1)   # [C, T, H, W]
    C, T, scene_h, scene_w = scene.shape
    assert C == len(BAND_ORDER) and T == NUM_FRAMES

    valid_px = int(np.sum(~scene_nodata))
    print(f"  Scene (C,T,H,W): {scene.shape}  Valid: {valid_px:,}/{scene_h*scene_w:,} px")

    chips = []
    for row in range(0, scene_h, stride):
        for col in range(0, scene_w, stride):
            h = min(chip_size, scene_h - row)
            w = min(chip_size, scene_w - col)
            chip_data = scene[:, :, row:row+h, col:col+w]
            chip_nd   = scene_nodata[row:row+h, col:col+w]
            if chip_nd.all():
                continue
            if h < chip_size or w < chip_size:
                pad    = np.zeros((C, T, chip_size, chip_size), dtype=np.float32)
                pad_nd = np.ones((chip_size, chip_size), dtype=bool)
                pad[:, :, :h, :w]  = chip_data
                pad_nd[:h, :w]     = chip_nd
                chip_data, chip_nd = pad, pad_nd
            chips.append({
                "tensor":      chip_data,
                "nodata_mask": chip_nd,
                "row_off":     row,
                "col_off":     col,
                "chip_h":      h,
                "chip_w":      w,
                "scene_h":     scene_h,
                "scene_w":     scene_w,
                "transform":   meta_ref["transform"],
                "crs":         meta_ref["crs"],
            })
    return chips


# RUN
print("=" * 60)
print("  CHIPPING INTO 224×224 TILES")
print("=" * 60)
chips = extract_chips(timestep_stacks)
print(f"[OK] {len(chips)} chips  shape: {chips[0]['tensor'].shape}  "
      f"scene: {chips[0]['scene_h']}×{chips[0]['scene_w']}px")
print("=" * 60)

  CHIPPING INTO 224×224 TILES
   ⚠️  T1 shape differs — reprojecting
   ⚠️  T2 shape differs — reprojecting
  Scene (C,T,H,W): (6, 3, 1422, 1350)  Valid: 1,702,114/1,919,700 px
[OK] 41 chips  shape: (6, 3, 224, 224)  scene: 1422×1350px


In [7]:
# =============================================================================
# STEP 5 — LOAD MODEL + RUN INFERENCE
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time as _time
from pathlib import Path
from huggingface_hub import hf_hub_download

# ── Config ────────────────────────────────────────────────────────────────────
BATCH_SIZE  = 4

# =============================================================================
# MODEL ARCHITECTURE
# =============================================================================

class PatchEmbed3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv3d(6, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16))

    def forward(self, x):
        x = self.proj(x)                              # (B, 768, T, H_p, W_p)
        B, E, T, H, W = x.shape
        return x.flatten(2).transpose(1, 2), T, H, W  # (B, T*H*W, 768), T, H, W


class Attention(nn.Module):
    def __init__(self, dim=768, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.scale     = (dim // num_heads) ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3)
        self.proj      = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        x = (q @ k.transpose(-2, -1) * self.scale).softmax(-1) @ v
        return self.proj(x.transpose(1, 2).reshape(B, N, C))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(768)
        self.attn  = Attention()
        self.norm2 = nn.LayerNorm(768)
        self.mlp   = nn.Sequential(
            nn.Linear(768, 3072),
            nn.GELU(),
            nn.Linear(3072, 768),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class LNWrapper(nn.Module):
    def __init__(self, dim=2304):
        super().__init__()
        self.ln = nn.LayerNorm(dim)

    def forward(self, x):
        B, C, H, W = x.shape
        return self.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)


class FPNBlock(nn.Module):
    def __init__(self, dim=2304):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(dim, dim, 2, stride=2),
            LNWrapper(dim),
            nn.GELU(),
            nn.ConvTranspose2d(dim, dim, 2, stride=2),
        )

    def forward(self, x):
        return self.block(x)


class ConvModule(nn.Module):
    def __init__(self, in_c, out_c, k, p=0):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, k, padding=p, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class AuxFCNHead(nn.Module):
    def __init__(self, in_c=2304, ch=256, nc=NUM_CLASSES, dr=HEAD_DROPOUT):
        super().__init__()
        self.convs    = nn.ModuleList([ConvModule(in_c, ch, 3, 1), ConvModule(ch, ch, 3, 1)])
        self.dropout  = nn.Dropout2d(dr)
        self.conv_seg = nn.Conv2d(ch, nc, 1)

    def forward(self, x):
        for c in self.convs:
            x = c(x)
        return self.conv_seg(self.dropout(x))


# FIX 1 — proper nn.Module subclasses for neck and decode_head
# bare nn.Module() instances cannot register submodules correctly,
# causing parameters to be invisible to .parameters() and .to(device)
class Neck(nn.Module):
    def __init__(self):
        super().__init__()
        self.fpn1 = FPNBlock()
        self.fpn2 = FPNBlock()

    def forward(self, x):
        return self.fpn1(x)


class DecodeHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs    = nn.ModuleList([ConvModule(2304, 256, 3, 1)])
        self.dropout  = nn.Dropout2d(HEAD_DROPOUT)
        self.conv_seg = nn.Conv2d(256, NUM_CLASSES, 1)

    def forward(self, x):
        for conv in self.convs:
            x = conv(x)
        return self.conv_seg(self.dropout(x))


class PrithviCropClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.patch_embed = PatchEmbed3D()
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, 768))
        self.pos_embed   = nn.Parameter(torch.zeros(1, 3 * 14 * 14 + 1, 768))
        self.blocks      = nn.ModuleList([Block() for _ in range(12)])
        self.norm        = nn.LayerNorm(768)
        self.neck        = Neck()         # FIX 1a
        self.decode_head = DecodeHead()   # FIX 1b
        self.auxiliary_head = AuxFCNHead()

    def _interpolate_pos_embed(self, T, H, W):
        cls_pos = self.pos_embed[:, :1, :]
        spa_pos = self.pos_embed[:, 1:, :]
        T_ref, H_ref, W_ref = 3, 14, 14
        spa_pos = (
            spa_pos
            .reshape(1, T_ref, H_ref, W_ref, 768)
            .permute(0, 4, 1, 2, 3)
        )
        spa_pos = (
            F.interpolate(
                spa_pos.reshape(1, 768, T_ref, H_ref * W_ref),
                size=(T, H * W),
                mode="bilinear",
                align_corners=False,
            )
            .reshape(1, 768, T, H, W)
            .permute(0, 2, 3, 4, 1)
            .reshape(1, T * H * W, 768)
        )
        return torch.cat([cls_pos, spa_pos], dim=1)

    def forward(self, x):
        tokens, T, H, W = self.patch_embed(x)
        B = tokens.shape[0]
        pos    = self._interpolate_pos_embed(T, H, W)
        tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1) + pos
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)
        feat = (
            tokens[:, 1:]
            .reshape(B, T, H, W, 768)
            .permute(0, 1, 4, 2, 3)
            .reshape(B, T * 768, H, W)
        )
        feat   = self.neck(feat)
        logits = self.decode_head(feat)
        return F.interpolate(
            logits,
            size=(x.shape[3], x.shape[4]),
            mode="bilinear",
            align_corners=False,
        )


# =============================================================================
# MODEL LOADER
# =============================================================================

def load_crop_model():
    print("=" * 60)
    print("  LOADING PRITHVI-EO-1.0-100M CROP CLASSIFICATION MODEL")
    print("=" * 60)

    local_weights = globals().get("CROP_MODEL_LOCAL", None)
    if local_weights and Path(local_weights).exists():
        ckpt_path = Path(local_weights)
        print(f"   Using fine-tuned weights : {ckpt_path}")
    else:
        print("   Downloading base weights from HuggingFace...")
        ckpt_path = Path(hf_hub_download(
            repo_id  = "ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification",
            filename = "multi_temporal_crop_classification_Prithvi_100M.pth",
        ))
        print(f"   Checkpoint : {ckpt_path}")

    model = PrithviCropClassifier()
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if "state_dict" in state:
        state = state["state_dict"]

    missing, unexpected = model.load_state_dict(state, strict=False)
    non_aux_missing = [k for k in missing if "auxiliary_head" not in k]
    print(f"   Missing (non-aux) : {len(non_aux_missing)}")
    if non_aux_missing:
        print(f"   ⚠️  {non_aux_missing[:3]}")

    model.eval()
    print(f"   ✅ Model ready ({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params)")
    print("=" * 60)
    return model


# =============================================================================
# LOAD MODEL
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

model = load_crop_model()
model = model.to(device)
model.eval()

# =============================================================================
# SANITY CHECK
# =============================================================================

print("\nRunning pos_embed sanity check...")
_dummy = torch.zeros(1, 6, 3, CHIP_SIZE, CHIP_SIZE).to(device)
with torch.no_grad():
    _out = model(_dummy)
print(f"[OK] Dummy forward: input {tuple(_dummy.shape)} → output {tuple(_out.shape)}")
del _dummy, _out

# =============================================================================
# INFERENCE
# =============================================================================

results   = []
n_batches = (len(chips) + BATCH_SIZE - 1) // BATCH_SIZE
t_start   = _time.time()

print(f"\nRunning inference on {len(chips)} chips (batch={BATCH_SIZE})...")

for batch_idx in range(n_batches):
    i0          = batch_idx * BATCH_SIZE
    i1          = min(i0 + BATCH_SIZE, len(chips))
    batch_chips = chips[i0:i1]

    # FIX 2 — chip["tensor"] is [C, T, H, W]; stack across batch dim → (B, C, T, H, W)
    x = np.stack([c["tensor"] for c in batch_chips], axis=0)  # (B, C, T, H, W)

    with torch.no_grad():
        logits   = model(torch.from_numpy(x).float().to(device))
    preds_np = torch.argmax(logits, dim=1).cpu().numpy()       # (B, H, W)

    for j, chip in enumerate(batch_chips):
        results.append({
            "pred":        preds_np[j],
            "nodata_mask": chip["nodata_mask"],
            "row_off":     chip["row_off"],
            "col_off":     chip["col_off"],
            "chip_h":      chip["chip_h"],
            "chip_w":      chip["chip_w"],
        })

    if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == n_batches:
        rate = (batch_idx + 1) * BATCH_SIZE / (_time.time() - t_start)
        print(f"  {i1}/{len(chips)} chips  {rate:.1f} chips/s")

preds = results
print(f"\n[OK] Inference complete — {len(preds)} chips in {_time.time() - t_start:.1f}s")

# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

all_preds = np.concatenate([
    r["pred"][:r["chip_h"], :r["chip_w"]][~r["nodata_mask"][:r["chip_h"], :r["chip_w"]]].reshape(-1)
    for r in preds
])

unique, counts = np.unique(all_preds, return_counts=True)
print("\nClass distribution:")
for cls, cnt in zip(unique, counts):
    name  = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"class_{cls}"
    acres = cnt * (900 / 4046.856)
    print(f"  {cls:>2}  {name:<22}: {cnt:>8,} px  (~{acres:>8,.0f} acres)")

Device : cuda
GPU    : NVIDIA A10G
  LOADING PRITHVI-EO-1.0-100M CROP CLASSIFICATION MODEL
   Checkpoint : /home/sagemaker-user/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-1.0-100M-multi-temporal-crop-classification/snapshots/b53a88b8da673800b67c34a98a527b77076e7035/multi_temporal_crop_classification_Prithvi_100M.pth
   Missing (non-aux) : 162
   ⚠️  ['cls_token', 'pos_embed', 'patch_embed.proj.weight']
   ✅ Model ready (182.9M params)

Running pos_embed sanity check...
[OK] Dummy forward: input (1, 6, 3, 224, 224) → output (1, 13, 224, 224)

Running inference on 41 chips (batch=4)...
  20/41 chips  70.1 chips/s
  40/41 chips  71.5 chips/s
  41/41 chips  76.4 chips/s

[OK] Inference complete — 41 chips in 0.6s

Class distribution:
   0  Corn                  :  149,706 px  (~  33,294 acres)
   1  Soybean               :   88,166 px  (~  19,608 acres)
   2  Spring Wheat          :  150,414 px  (~  33,451 acres)
   3  Winter Wheat          :      298 px  (~      66 acr

In [8]:
# =============================================================================
# STEP 6 — REASSEMBLE CHIP PREDICTIONS INTO FULL-SCENE MOSAIC
# =============================================================================

import numpy as np
import rasterio
from pathlib import Path
from scipy.ndimage import binary_opening, binary_closing

# ── Config ────────────────────────────────────────────────────────────────────
CORN_CLASS_IDX   = 0
MORPH_OPEN_SIZE  = 3
MORPH_CLOSE_SIZE = 5

_default_threshold = 0.60
CORN_THRESHOLD = float(globals().get("CORN_THRESHOLD", _default_threshold))
print(f"[INFO] Using CORN_THRESHOLD = {CORN_THRESHOLD:.2f} "
      f"({'from Step 7.5 sweep' if 'CORN_THRESHOLD' in globals() else 'default'})")

# ── Derive scene dimensions from chip metadata ────────────────────────────────
scene_h = max(r["row_off"] + r["chip_h"] for r in preds)
scene_w = max(r["col_off"] + r["chip_w"] for r in preds)

print(f"[INFO] Assembling mosaic: {scene_h}×{scene_w} px from {len(preds)} chips")
print(f"       Blending: one-hot argmax voting")

# =============================================================================
# PART A — ONE-HOT VOTING
# Step 5 stored argmax only (no logits), so we accumulate per-class votes.
# =============================================================================

vote_accum = np.zeros((NUM_CLASSES, scene_h, scene_w), dtype=np.float32)
weight_map = np.zeros((scene_h, scene_w), dtype=np.float32)

for r in preds:
    r0, c0  = r["row_off"], r["col_off"]
    h,  w   = r["chip_h"],  r["chip_w"]
    valid   = ~r["nodata_mask"][:h, :w]
    pred_hw = r["pred"][:h, :w]

    for cls in range(NUM_CLASSES):
        mask = valid & (pred_hw == cls)
        vote_accum[cls, r0:r0+h, c0:c0+w][mask] += 1.0
    weight_map[r0:r0+h, c0:c0+w][valid] += 1.0

covered    = weight_map > 0
prob_accum = vote_accum.copy()
prob_accum[:, covered] /= weight_map[covered]

# =============================================================================
# PART B — APPLY CORN THRESHOLD + ARGMAX
# =============================================================================

prob_accum_thresholded = prob_accum.copy()
prob_accum_thresholded[CORN_CLASS_IDX][prob_accum[CORN_CLASS_IDX] < CORN_THRESHOLD] = 0.0

scene_pred = np.full((scene_h, scene_w), -1, dtype=np.int16)
scene_pred[covered] = np.argmax(
    prob_accum_thresholded[:, covered], axis=0
).astype(np.int16)

corn_px_before = int((scene_pred == CORN_CLASS_IDX).sum())
print(f"[INFO] Corn pixels before morphological cleanup : {corn_px_before:,}")

# =============================================================================
# PART C — MORPHOLOGICAL CLEANUP ON CORN MASK
# =============================================================================

corn_mask    = (scene_pred == CORN_CLASS_IDX)
struct_open  = np.ones((MORPH_OPEN_SIZE,  MORPH_OPEN_SIZE),  dtype=bool)
struct_close = np.ones((MORPH_CLOSE_SIZE, MORPH_CLOSE_SIZE), dtype=bool)

corn_mask_clean = binary_opening(corn_mask,       structure=struct_open)
corn_mask_clean = binary_closing(corn_mask_clean, structure=struct_close)

corn_removed = corn_mask & ~corn_mask_clean
corn_added   = corn_mask_clean & ~corn_mask

if corn_removed.any():
    # FIX — argsort returns (NUM_CLASSES, N); [-2] picks second-best per pixel
    second_best = np.argsort(prob_accum[:, corn_removed], axis=0)[-2, :]
    scene_pred[corn_removed] = second_best.astype(np.int16)

scene_pred[corn_added] = CORN_CLASS_IDX
scene_pred[~covered]   = -1

corn_px_after = int((scene_pred == CORN_CLASS_IDX).sum())
print(f"[INFO] Corn pixels after  morphological cleanup : {corn_px_after:,}")
print(f"       Removed : {corn_px_before - corn_px_after + int(corn_added.sum()):,} px  |  "
      f"Filled : {int(corn_added.sum()):,} px")

# =============================================================================
# PART D — WRITE GEOTIFF
# =============================================================================

scene_transform = preds[0].get("transform", chips[0]["transform"])
scene_crs       = preds[0].get("crs",       chips[0]["crs"])

PRED_DIR = Path("./polk_corn_validation/predictions")
PRED_DIR.mkdir(parents=True, exist_ok=True)
pred_raster_path = PRED_DIR / "scene_pred.tif"

with rasterio.open(
    pred_raster_path, "w",
    driver    = "GTiff",
    height    = scene_pred.shape[0],
    width     = scene_pred.shape[1],
    count     = 1,
    dtype     = np.int16,
    crs       = scene_crs,
    transform = scene_transform,
    nodata    = -1,
) as dst:
    dst.write(scene_pred.astype(np.int16), 1)

print(f"\n[OK] Scene prediction : {scene_pred.shape}  "
      f"Covered: {covered.sum():,}/{scene_pred.size:,} px")
print(f"[OK] Saved : {pred_raster_path}")

# =============================================================================
# PART E — CLASS DISTRIBUTION SUMMARY
# =============================================================================

print("\nClass distribution (post-cleanup):")
unique, counts = np.unique(scene_pred[covered], return_counts=True)
for cls, cnt in zip(unique, counts):
    name  = CLASS_NAMES[int(cls)] if int(cls) < len(CLASS_NAMES) else f"class_{cls}"
    acres = cnt * (900 / 4046.856)
    pct   = 100 * cnt / covered.sum()
    print(f"  {cls:>2}  {name:<22}: {cnt:>8,} px  (~{acres:>8,.0f} acres)  {pct:>5.1f}%")

[INFO] Using CORN_THRESHOLD = 0.60 (from Step 7.5 sweep)
[INFO] Assembling mosaic: 1422×1350 px from 41 chips
       Blending: one-hot argmax voting
[INFO] Corn pixels before morphological cleanup : 149,706
[INFO] Corn pixels after  morphological cleanup : 129,007
       Removed : 32,977 px  |  Filled : 12,278 px

[OK] Scene prediction : (1422, 1350)  Covered: 1,702,114/1,919,700 px
[OK] Saved : polk_corn_validation/predictions/scene_pred.tif

Class distribution (post-cleanup):
   0  Corn                  :  129,007 px  (~  28,690 acres)    7.6%
   1  Soybean               :   87,203 px  (~  19,393 acres)    5.1%
   2  Spring Wheat          :  148,828 px  (~  33,099 acres)    8.7%
   3  Winter Wheat          :      298 px  (~      66 acres)    0.0%
   4  Rice                  :   70,871 px  (~  15,761 acres)    4.2%
   5  Sorghum               :  128,624 px  (~  28,605 acres)    7.6%
   6  Other Small Grain     :      385 px  (~      86 acres)    0.0%
   7  Rye                   :  535

In [9]:
# =============================================================================
# STEP 7 — CDL VALIDATION
# =============================================================================

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject as rio_reproject_val

# ── Constants ─────────────────────────────────────────────────────────────────
ACRES_PER_PIXEL = 900 / 4046.856
CDL_CORN_VALUE  = 1
CORN_CLASS_IDX  = 0

CDL_NAMES = {
    0:   "No Data / Unclassified",
    1:   "Corn",
    5:   "Soybeans",
    22:  "Spring Wheat",
    24:  "Winter Wheat",
    3:   "Rice",
    4:   "Sorghum",
    27:  "Rye",
    28:  "Oats",
    2:   "Cotton",
    31:  "Canola",
    36:  "Barley",
    37:  "Other Hay/Non Alfalfa",
    111: "Open Water",
    121: "Developed/Open Space",
    122: "Developed/Low Intensity",
    131: "Barren",
    141: "Deciduous Forest",
    142: "Evergreen Forest",
    143: "Mixed Forest",
    152: "Shrubland",
    176: "Grassland/Pasture",
    190: "Woody Wetlands",
    195: "Herbaceous Wetlands",
}

print("=" * 60)
print("  CDL VALIDATION")
print("=" * 60)

# Guard — ensure upstream steps ran
if "scene_pred" not in dir() and "scene_pred" not in globals():
    raise RuntimeError("scene_pred not found — run Step 6 first.")
if "cdl_path" not in globals():
    raise RuntimeError("cdl_path not found — run Step 3 first.")

# FIX — use consistent output path from Step 6
pred_raster_path = OUT_DIR / "predictions" / "scene_pred.tif"
if not pred_raster_path.exists():
    raise RuntimeError(f"Prediction raster not found: {pred_raster_path}")

# =============================================================================
# PART A — REPROJECT CDL TO MATCH PREDICTION RASTER
# =============================================================================

with rasterio.open(pred_raster_path) as pred_ds:
    pred_crs       = pred_ds.crs
    pred_transform = pred_ds.transform
    pred_h         = pred_ds.height
    pred_w         = pred_ds.width

with rasterio.open(cdl_path) as cdl_ds:
    cdl_aligned = np.zeros((pred_h, pred_w), dtype=np.uint8)
    rio_reproject_val(
        source        = rasterio.band(cdl_ds, 1),
        destination   = cdl_aligned,
        src_transform = cdl_ds.transform,
        src_crs       = cdl_ds.crs,
        dst_transform = pred_transform,
        dst_crs       = pred_crs,
        resampling    = Resampling.nearest,
    )

print(f"   CDL aligned: {cdl_aligned.shape}")

# =============================================================================
# PART B — BINARY CORN MASKS + VALID PIXEL MASK
# =============================================================================

pred_corn_mask = (scene_pred == CORN_CLASS_IDX).astype(np.uint8)
cdl_corn_mask  = (cdl_aligned == CDL_CORN_VALUE).astype(np.uint8)
valid_mask     = (scene_pred >= 0) & (cdl_aligned > 0)

pred_flat = pred_corn_mask[valid_mask]
cdl_flat  = cdl_corn_mask[valid_mask]

valid_px = valid_mask.sum()
print(f"   Valid pixels for comparison: {valid_px:,} ({100*valid_px/scene_pred.size:.1f}% of scene)")

# =============================================================================
# PART C — CONFUSION MATRIX + METRICS
# =============================================================================

TP = int(np.sum((pred_flat == 1) & (cdl_flat == 1)))
FP = int(np.sum((pred_flat == 1) & (cdl_flat == 0)))
FN = int(np.sum((pred_flat == 0) & (cdl_flat == 1)))
TN = int(np.sum((pred_flat == 0) & (cdl_flat == 0)))

precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)
iou       = TP / (TP + FP + FN + 1e-9)
pixel_acc = (TP + TN) / (TP + FP + FN + TN + 1e-9)

pred_corn_acres = int(pred_flat.sum()) * ACRES_PER_PIXEL
cdl_corn_acres  = int(cdl_flat.sum())  * ACRES_PER_PIXEL
acre_diff       = pred_corn_acres - cdl_corn_acres
pct_diff        = acre_diff / (cdl_corn_acres + 1) * 100

# =============================================================================
# PART D — MAIN RESULTS BLOCK
# =============================================================================

print()
print("=" * 60)
print("  CORN CLASSIFICATION RESULTS — POLK COUNTY, IOWA")
print("=" * 60)
print(f"\n  Pixel-level metrics (corn vs. non-corn):")
print(f"    True Positives  : {TP:>10,}  px  (corn correctly identified)")
print(f"    False Positives : {FP:>10,}  px  (predicted corn, actually not corn)")
print(f"    False Negatives : {FN:>10,}  px  (corn missed by model)")
print(f"    True Negatives  : {TN:>10,}  px  (non-corn correctly identified)")
print(f"\n  Performance:")
print(f"    Precision (corn) : {precision:.3f}")
print(f"    Recall    (corn) : {recall:.3f}")
print(f"    F1 Score  (corn) : {f1:.3f}")
print(f"    IoU       (corn) : {iou:.3f}")
print(f"    Pixel accuracy   : {pixel_acc:.3f}")
print(f"\n  Area estimates:")
print(f"    Model predicted corn : {pred_corn_acres:>10,.0f} acres")
print(f"    CDL ground-truth corn: {cdl_corn_acres:>10,.0f} acres")

sign      = "+" if acre_diff >= 0 else "-"
direction = "overcount" if acre_diff >= 0 else "undercount"
print(f"    Area difference      : {sign}{abs(acre_diff):>8,.0f} acres "
      f"({sign}{abs(pct_diff):.1f}%  {direction})")

# =============================================================================
# PART E — PER-CLASS FALSE POSITIVE BREAKDOWN
# =============================================================================

print(f"\n  False Positive breakdown (what the model called corn but isn't):")
print(f"  {'CDL Class':<28} {'FP pixels':>10}  {'% of FPs':>9}")
print(f"  {'-'*50}")

fp_mask              = (pred_flat == 1) & (cdl_flat == 0)
cdl_at_fp            = cdl_aligned[valid_mask][fp_mask]
fp_unique, fp_counts = np.unique(cdl_at_fp, return_counts=True)
fp_sort              = np.argsort(fp_counts)[::-1]

for cdl_val, cnt in zip(fp_unique[fp_sort], fp_counts[fp_sort]):
    name = CDL_NAMES.get(int(cdl_val), f"CDL code {cdl_val}")
    pct  = 100 * cnt / (FP + 1e-9)
    print(f"  {name:<28} {cnt:>10,}  {pct:>8.1f}%")

# =============================================================================
# PART F — FALSE NEGATIVE BREAKDOWN
# =============================================================================

print(f"\n  False Negative breakdown (corn the model missed):")
print(f"    {FN:,} px missed  "
      f"({100*FN/(TP+FN+1e-9):.1f}% of actual corn not detected)")

# =============================================================================
# PART G — VERDICT
# =============================================================================

print()
if f1 >= 0.85:
    verdict = "✅  Strong detection — production ready"
elif f1 >= 0.75:
    verdict = "✅  Good detection — minor threshold tuning may help"
elif f1 >= 0.65:
    verdict = "⚠️   Moderate detection — re-run Step 7.5 with corrected weights"
elif f1 >= 0.50:
    verdict = "⚠️   Weak detection — check chip coverage and class weights"
else:
    verdict = "❌  Low detection — review imagery quality and fine-tuning setup"

print(f"  {verdict}")
print("=" * 60)

# cdl_aligned remains in scope for Step 7.5 label attachment

  CDL VALIDATION
   CDL aligned: (1422, 1350)
   Valid pixels for comparison: 1,701,422 (88.6% of scene)

  CORN CLASSIFICATION RESULTS — POLK COUNTY, IOWA

  Pixel-level metrics (corn vs. non-corn):
    True Positives  :     13,041  px  (corn correctly identified)
    False Positives :    115,922  px  (predicted corn, actually not corn)
    False Negatives :    328,234  px  (corn missed by model)
    True Negatives  :  1,244,225  px  (non-corn correctly identified)

  Performance:
    Precision (corn) : 0.101
    Recall    (corn) : 0.038
    F1 Score  (corn) : 0.055
    IoU       (corn) : 0.029
    Pixel accuracy   : 0.739

  Area estimates:
    Model predicted corn :     28,681 acres
    CDL ground-truth corn:     75,898 acres
    Area difference      : -  47,217 acres (-62.2%  undercount)

  False Positive breakdown (what the model called corn but isn't):
  CDL Class                     FP pixels   % of FPs
  --------------------------------------------------
  Developed/Low Intensi

In [10]:
# =============================================================================
# STEP 7.5 — LOCAL FINE-TUNING
# =============================================================================

import time
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import binary_opening, binary_closing

# =============================================================================
# CONFIG
# =============================================================================

FINETUNE_EPOCHS     = 120
FINETUNE_LR         = 3e-5
FINETUNE_BATCH      = 4
EARLY_STOP_PATIENCE = 15

CORN_CLASS_IDX    = 0
NONCROP_CLASS_IDX = 7
NUM_CLASSES       = 13

THRESHOLD_SWEEP = np.arange(0.30, 0.91, 0.02)

MORPH_OPEN_SIZE  = 3
MORPH_CLOSE_SIZE = 5

FORCE_RESTART = True

LOCAL_CKPT_DIR  = Path("./checkpoints")
LOCAL_CKPT_PATH = LOCAL_CKPT_DIR / "latest_checkpoint.pt"
FINAL_WEIGHTS   = Path("./trained_model/prithvi_crop_finetuned.pt")

LOCAL_CKPT_DIR.mkdir(exist_ok=True)
FINAL_WEIGHTS.parent.mkdir(exist_ok=True)

# Guard — upstream dependencies
for _var, _step in [("chips", "Step 4"), ("cdl_aligned", "Step 7"), ("load_crop_model", "Step 5")]:
    if _var not in globals():
        raise RuntimeError(f"'{_var}' not found — run {_step} first.")

# =============================================================================
# CHECKPOINT GUARD
# =============================================================================

if LOCAL_CKPT_PATH.exists():
    if FORCE_RESTART:
        LOCAL_CKPT_PATH.unlink()
        print("[INFO] FORCE_RESTART=True — deleted existing checkpoint.")
    else:
        ckpt_check = torch.load(LOCAL_CKPT_PATH, map_location="cpu")
        last_epoch = ckpt_check.get("epoch", -1)
        if last_epoch >= FINETUNE_EPOCHS - 1:
            LOCAL_CKPT_PATH.unlink()
            print(f"[INFO] Completed checkpoint (epoch={last_epoch}) deleted — starting fresh.")
        else:
            print(f"[RESUME] Mid-run checkpoint found at epoch {last_epoch} — will resume.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# =============================================================================
# PART A — CDL -> PRITHVI LABEL MAPPING
# =============================================================================

CDL_TO_PRITHVI = {
    1:   0,   # Corn
    5:   1,   # Soybeans
    22:  2,   # Spring Wheat
    24:  3,   # Winter Wheat
    3:   4,   # Rice
    4:   5,   # Sorghum
    27:  6,   # Rye
    28:  8,   # Oats
    2:   9,   # Cotton
    31:  10,  # Canola
    36:  12,  # Barley
    # Non-crop -> class 7 (background)
    176: 7,   # Grassland/Pasture
    121: 7,   # Developed/Open Space
    122: 7,   # Developed/Low Intensity
    123: 7,   # Developed/Medium Intensity
    124: 7,   # Developed/High Intensity
    141: 7,   # Deciduous Forest
    142: 7,   # Evergreen Forest
    143: 7,   # Mixed Forest
    111: 7,   # Open Water
    190: 7,   # Woody Wetlands
    195: 7,   # Herbaceous Wetlands
    131: 7,   # Barren
    152: 7,   # Shrubland
    37:  7,   # Other Hay/Non Alfalfa
    59:  7,   # Sod/Grass Seed
    60:  7,   # Switchgrass
    61:  7,   # Fallow/Idle Cropland
    58:  7,   # Clover/Wildflowers
    44:  7,   # Other Crops
    12:  7,   # Sweet Corn
    13:  7,   # Pop/Orn Corn
}

# =============================================================================
# PART A1 — SPECTRAL INDEX HELPERS
# =============================================================================

BAND_BLUE    = 0
BAND_GREEN   = 1
BAND_RED     = 2
BAND_REDEDGE = 3
BAND_NIR     = 4
BAND_SWIR    = 5

T_EARLY = 0
T_PEAK  = 1
T_LATE  = 2
N_TIMES = 3

def compute_spectral_indices(tensor):
    """tensor: np.ndarray (6, 3, H, W) — bands axis 0, timesteps axis 1."""
    t   = tensor.astype(np.float32)
    eps = 1e-6

    def safe_ratio(a, b):
        return np.clip((a - b) / (a + b + eps), -1.0, 1.0)

    ndvi = [safe_ratio(t[BAND_NIR, ti], t[BAND_RED, ti]) for ti in range(N_TIMES)]
    ndvi_peak_late_diff = ndvi[T_PEAK] - ndvi[T_LATE]

    ndwi_mean = np.mean([
        safe_ratio(t[BAND_GREEN, ti], t[BAND_NIR, ti]) for ti in range(N_TIMES)
    ], axis=0)

    lswi      = [safe_ratio(t[BAND_NIR, ti], t[BAND_SWIR, ti]) for ti in range(N_TIMES)]
    lswi_mean = np.mean(lswi, axis=0)
    lswi_peak = lswi[T_PEAK]

    re_ratio_peak    = np.clip(t[BAND_REDEDGE, T_PEAK] / (t[BAND_NIR, T_PEAK] + eps), 0.0, 2.0)
    nir_temporal_var = np.var(t[BAND_NIR], axis=0)

    return {
        "ndvi_peak_late_diff": ndvi_peak_late_diff,
        "ndwi_mean":           ndwi_mean,
        "lswi_mean":           lswi_mean,
        "lswi_peak":           lswi_peak,
        "re_ratio_peak":       re_ratio_peak,
        "nir_temporal_var":    nir_temporal_var,
    }

# =============================================================================
# PART A2 — ATTACH CDL LABELS TO CHIPS
# =============================================================================

print("Attaching CDL labels to chips...")

labeled_chips = []
label_stats   = {"corn": 0, "other_crop": 0, "non_crop": 0, "ignored": 0}

for chip in chips:
    r0, c0 = chip["row_off"], chip["col_off"]
    h,  w  = chip["chip_h"],  chip["chip_w"]

    cdl_crop   = cdl_aligned[r0:r0+h, c0:c0+w]
    chip_label = np.full((h, w), fill_value=-1, dtype=np.int64)

    for cdl_val, prithvi_cls in CDL_TO_PRITHVI.items():
        chip_label[cdl_crop == cdl_val] = prithvi_cls

    chip_label[chip["nodata_mask"][:h, :w]] = -1

    chip_H, chip_W = chip["tensor"].shape[2], chip["tensor"].shape[3]
    label = np.full((chip_H, chip_W), fill_value=-1, dtype=np.int64)
    label[:h, :w] = chip_label

    label_stats["corn"]       += int((chip_label == CORN_CLASS_IDX).sum())
    label_stats["non_crop"]   += int((chip_label == NONCROP_CLASS_IDX).sum())
    label_stats["other_crop"] += int(
        ((chip_label >= 0) &
         (chip_label != CORN_CLASS_IDX) &
         (chip_label != NONCROP_CLASS_IDX)).sum()
    )
    label_stats["ignored"] += int((chip_label == -1).sum())

    spectral = compute_spectral_indices(chip["tensor"])
    labeled_chips.append({**chip, **spectral, "label": label})

total_labeled = label_stats["corn"] + label_stats["other_crop"] + label_stats["non_crop"]
print(f"[OK] {len(labeled_chips)} chips labeled")
print(f"     Corn px      : {label_stats['corn']:>10,}  ({100*label_stats['corn']/max(total_labeled,1):.1f}%)")
print(f"     Non-crop px  : {label_stats['non_crop']:>10,}  ({100*label_stats['non_crop']/max(total_labeled,1):.1f}%)")
print(f"     Other crop px: {label_stats['other_crop']:>10,}  ({100*label_stats['other_crop']/max(total_labeled,1):.1f}%)")
print(f"     Ignored px   : {label_stats['ignored']:>10,}  (nodata)")

# =============================================================================
# PART B — DATASET + DATALOADERS
# FIX — num_workers=0 avoids deadlocks in Jupyter/SageMaker notebook kernels
# =============================================================================

class InMemoryChipDataset(Dataset):
    def __init__(self, chip_list):
        self.chips = chip_list

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, idx):
        c      = self.chips[idx]
        tensor = torch.from_numpy(c["tensor"].copy()).float()
        label  = torch.from_numpy(c["label"].copy()).long()

        if torch.rand(1).item() > 0.5:
            tensor = torch.flip(tensor, dims=[-1])
            label  = torch.flip(label,  dims=[-1])
        if torch.rand(1).item() > 0.5:
            tensor = torch.flip(tensor, dims=[-2])
            label  = torch.flip(label,  dims=[-2])
        k = torch.randint(0, 4, (1,)).item()
        if k > 0:
            tensor = torch.rot90(tensor, k=k, dims=[-2, -1])
            label  = torch.rot90(label,  k=k, dims=[-2, -1])
        if torch.rand(1).item() > 0.5:
            tensor = tensor + torch.randn_like(tensor) * 0.01

        return tensor, label

random.seed(42)
shuffled  = labeled_chips.copy()
random.shuffle(shuffled)

split_idx     = int(len(shuffled) * 0.8)
train_dataset = InMemoryChipDataset(shuffled[:split_idx])
val_dataset   = InMemoryChipDataset(shuffled[split_idx:])

# FIX — num_workers=0: multiprocessing workers cause deadlocks in notebooks
_nw = 0
train_loader = DataLoader(train_dataset, batch_size=FINETUNE_BATCH, shuffle=True,
                          num_workers=_nw, pin_memory=(device.type == "cuda"))
val_loader   = DataLoader(val_dataset,   batch_size=FINETUNE_BATCH, shuffle=False,
                          num_workers=_nw, pin_memory=(device.type == "cuda"))

print(f"\nTrain: {len(train_dataset)} chips  |  Val: {len(val_dataset)} chips")

# =============================================================================
# PART C — MODEL + OPTIMIZER
# =============================================================================

CLASS_WEIGHTS = [
    1.00,   #  0 — Corn
    1.80,   #  1 — Soybean
    1.00,   #  2 — Spring Wheat
    1.00,   #  3 — Winter Wheat
    1.00,   #  4 — Rice
    1.00,   #  5 — Sorghum
    1.00,   #  6 — Rye
    1.50,   #  7 — Non-crop/Background
    1.00,   #  8 — Oats
    1.00,   #  9 — Cotton
    1.00,   # 10 — Canola
    1.00,   # 11 — (unused slot — kept so tensor length == NUM_CLASSES)
    1.00,   # 12 — Barley
]
assert len(CLASS_WEIGHTS) == NUM_CLASSES, \
    f"CLASS_WEIGHTS length {len(CLASS_WEIGHTS)} != NUM_CLASSES {NUM_CLASSES}"

finetune_model = load_crop_model()
finetune_model = finetune_model.to(device)

class_weights_tensor = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=-1)

backbone_params, head_params = [], []
for name, param in finetune_model.named_parameters():
    if any(k in name for k in ["patch_embed", "blocks", "norm", "cls_token", "pos_embed"]):
        backbone_params.append(param)
    else:
        head_params.append(param)

print(f"\nBackbone params : {sum(p.numel() for p in backbone_params)/1e6:.1f}M  @ LR={FINETUNE_LR/10:.1e}")
print(f"Head params     : {sum(p.numel() for p in head_params)/1e6:.1f}M  @ LR={FINETUNE_LR:.1e}")

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": FINETUNE_LR / 10},
    {"params": head_params,     "lr": FINETUNE_LR},
], weight_decay=1e-2)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)

# =============================================================================
# CHECKPOINT RESUME — after model is defined
# =============================================================================

start_epoch = 0
if LOCAL_CKPT_PATH.exists():
    ckpt       = torch.load(LOCAL_CKPT_PATH, map_location=device)
    last_epoch = ckpt.get("epoch", -1)
    if last_epoch < FINETUNE_EPOCHS - 1:
        finetune_model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = last_epoch + 1
        print(f"[RESUME] Resuming from epoch {start_epoch}/{FINETUNE_EPOCHS}")
    else:
        print(f"[INFO] Checkpoint epoch={last_epoch} complete — starting fresh.")

# =============================================================================
# PART D — TRAINING LOOP
# =============================================================================

def compute_val_f1(model, loader, device, corn_idx=CORN_CLASS_IDX):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds  = model(inputs).argmax(dim=1)
            valid  = labels != -1
            p_corn = (preds  == corn_idx) & valid
            l_corn = (labels == corn_idx) & valid
            tp += int((p_corn &  l_corn).sum())
            fp += int((p_corn & ~l_corn).sum())
            fn += int((~p_corn & l_corn).sum())
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    return 2 * prec * rec / (prec + rec + 1e-9), prec, rec

print("\n" + "=" * 60)
print(f"  FINE-TUNING — {FINETUNE_EPOCHS} EPOCHS  |  Device: {device}")
print("=" * 60)

best_val_f1      = 0.0
patience_counter = 0
t_total          = time.time()

for epoch in range(start_epoch, FINETUNE_EPOCHS):
    finetune_model.train()
    train_loss = 0.0
    t_epoch    = time.time()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(finetune_model(inputs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_train = train_loss / max(len(train_loader), 1)

    finetune_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_loss += criterion(
                finetune_model(val_inputs.to(device)),
                val_labels.to(device)
            ).item()
    avg_val = val_loss / max(len(val_loader), 1)

    val_f1, val_prec, val_rec = compute_val_f1(finetune_model, val_loader, device)
    epoch_time = time.time() - t_epoch
    eta        = epoch_time * (FINETUNE_EPOCHS - epoch - 1)

    torch.save({
        "epoch":                epoch,
        "model_state_dict":     finetune_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss":                 avg_train,
        "val_f1":               val_f1,
    }, LOCAL_CKPT_PATH)

    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        patience_counter = 0
        torch.save(finetune_model.state_dict(), FINAL_WEIGHTS)
        print(f"Epoch {epoch+1:>3}/{FINETUNE_EPOCHS} | "
              f"Loss {avg_train:.4f}/{avg_val:.4f} | "
              f"F1 {val_f1:.3f} (P={val_prec:.3f} R={val_rec:.3f}) | "
              f"{epoch_time:.0f}s | ETA {eta/60:.0f}min  ★ best")
    else:
        patience_counter += 1
        print(f"Epoch {epoch+1:>3}/{FINETUNE_EPOCHS} | "
              f"Loss {avg_train:.4f}/{avg_val:.4f} | "
              f"F1 {val_f1:.3f} (P={val_prec:.3f} R={val_rec:.3f}) | "
              f"{epoch_time:.0f}s | ETA {eta/60:.0f}min "
              f"(patience {patience_counter}/{EARLY_STOP_PATIENCE})")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"[EARLY STOP] Val F1 flat for {EARLY_STOP_PATIENCE} epochs.")
            break

print("=" * 60)
print(f"[OK] Training complete in {(time.time()-t_total)/60:.1f} min")
print(f"[OK] Best val F1 : {best_val_f1:.3f}")
print(f"[OK] Best weights: {FINAL_WEIGHTS}")

# =============================================================================
# PART E — THRESHOLD SWEEP ON VALIDATION SET
# =============================================================================

print("\n" + "=" * 60)
print("  THRESHOLD SWEEP — optimising corn class F1")
print("=" * 60)

finetune_model.eval()
all_probs, all_labels_list = [], []

with torch.no_grad():
    for val_inputs, val_labels in val_loader:
        logits    = finetune_model(val_inputs.to(device))
        probs     = torch.softmax(logits, dim=1)
        corn_prob = probs[:, CORN_CLASS_IDX].cpu().numpy().ravel()
        labels_np = val_labels.numpy().ravel()
        valid     = labels_np != -1
        all_probs.append(corn_prob[valid])
        all_labels_list.append(labels_np[valid])

all_probs_np  = np.concatenate(all_probs)
all_labels_np = np.concatenate(all_labels_list)
binary_gt     = (all_labels_np == CORN_CLASS_IDX).astype(np.int32)

print(f"Val set: {binary_gt.sum():,} corn px / {len(binary_gt):,} total "
      f"({100*binary_gt.mean():.1f}% corn)")
print(f"\n{'Threshold':>10} | {'Precision':>10} | {'Recall':>8} | {'F1':>8}")
print("-" * 45)

best_f1, best_threshold = 0.0, 0.50

for t in THRESHOLD_SWEEP:
    preds_t = (all_probs_np >= t).astype(np.int32)
    tp = int(((preds_t == 1) & (binary_gt == 1)).sum())
    fp = int(((preds_t == 1) & (binary_gt == 0)).sum())
    fn = int(((preds_t == 0) & (binary_gt == 1)).sum())
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    marker = "  <- best" if f1 > best_f1 else ""
    print(f"{t:>10.2f} | {prec:>10.3f} | {rec:>8.3f} | {f1:>8.3f}{marker}")
    if f1 > best_f1:
        best_f1, best_threshold = f1, float(t)

CORN_THRESHOLD = best_threshold
print(f"\n[OK] Best corn threshold = {CORN_THRESHOLD:.2f}  (val F1 = {best_f1:.3f})")

# =============================================================================
# PART F — MORPHOLOGICAL CLEANUP HELPER
# =============================================================================

def apply_morph_cleanup(corn_mask, open_size=MORPH_OPEN_SIZE, close_size=MORPH_CLOSE_SIZE):
    s_open  = np.ones((open_size,  open_size),  dtype=bool)
    s_close = np.ones((close_size, close_size), dtype=bool)
    cleaned = binary_opening(corn_mask.astype(bool), structure=s_open)
    cleaned = binary_closing(cleaned,               structure=s_close)
    return cleaned

print(f"\n[OK] apply_morph_cleanup() ready  "
      f"(open={MORPH_OPEN_SIZE}px, close={MORPH_CLOSE_SIZE}px)")

# =============================================================================
# PART G — HAND OFF TO STEPS 5/6/7
# =============================================================================

CROP_MODEL_LOCAL = str(FINAL_WEIGHTS)
print(f"\n{'='*60}")
print(f"[NEXT] CROP_MODEL_LOCAL = '{CROP_MODEL_LOCAL}'")
print(f"[NEXT] CORN_THRESHOLD   = {CORN_THRESHOLD:.2f}")
print(f"[NEXT] Re-run Steps 5 -> 6 -> 7 with fine-tuned weights.")
print(f"{'='*60}")

Device : cuda
GPU    : NVIDIA A10G
Attaching CDL labels to chips...
[OK] 41 chips labeled
     Corn px      :    341,275  (20.1%)
     Non-crop px  :  1,050,370  (61.7%)
     Other crop px:    309,768  (18.2%)
     Ignored px   :    143,023  (nodata)

Train: 32 chips  |  Val: 9 chips
  LOADING PRITHVI-EO-1.0-100M CROP CLASSIFICATION MODEL
   Checkpoint : /home/sagemaker-user/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-1.0-100M-multi-temporal-crop-classification/snapshots/b53a88b8da673800b67c34a98a527b77076e7035/multi_temporal_crop_classification_Prithvi_100M.pth
   Missing (non-aux) : 162
   ⚠️  ['cls_token', 'pos_embed', 'patch_embed.proj.weight']
   ✅ Model ready (182.9M params)

Backbone params : 86.7M  @ LR=3.0e-06
Head params     : 96.2M  @ LR=3.0e-05

  FINE-TUNING — 120 EPOCHS  |  Device: cuda
Epoch   1/120 | Loss 2.0392/5.6815 | F1 0.294 (P=0.372 R=0.243) | 3s | ETA 5min  ★ best
Epoch   2/120 | Loss 1.5072/3.6951 | F1 0.339 (P=0.459 R=0.268) | 2s | ETA 4min  

In [12]:
# =============================================================================
# STEP 7.6 — RE-RUN INFERENCE + VALIDATION WITH FINE-TUNED WEIGHTS
# =============================================================================

import time as _time
import csv

# Guard — upstream dependencies
for _var, _step in [
    ("chips",           "Step 4"),
    ("cdl_aligned",     "Step 7"),
    ("load_crop_model", "Step 5"),
    ("CROP_MODEL_LOCAL","Step 7.5"),
    ("pred_raster_path","Step 6"),
]:
    if _var not in globals():
        raise RuntimeError(f"'{_var}' not found — run {_step} first.")

print("=" * 60)
print("  STEP 7.6 — RE-RUNNING PIPELINE WITH FINE-TUNED WEIGHTS")
print("=" * 60)
print(f"  Loading weights from: {CROP_MODEL_LOCAL}")
print()

# =============================================================================
# PART A — RELOAD MODEL WITH FINE-TUNED WEIGHTS
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = load_crop_model()
model  = model.to(device)
model.eval()

# =============================================================================
# PART B — RE-RUN INFERENCE
# =============================================================================

BATCH_SIZE = 4
results    = []
n_batches  = (len(chips) + BATCH_SIZE - 1) // BATCH_SIZE
t_start    = _time.time()

print(f"Running inference on {len(chips)} chips (batch={BATCH_SIZE})...")

for batch_idx in range(n_batches):
    i0          = batch_idx * BATCH_SIZE
    i1          = min(i0 + BATCH_SIZE, len(chips))
    batch_chips = chips[i0:i1]

    x = np.stack([c["tensor"] for c in batch_chips], axis=0)
    with torch.no_grad():
        logits   = model(torch.from_numpy(x).float().to(device))
    preds_np = torch.argmax(logits, dim=1).cpu().numpy()

    for j, chip in enumerate(batch_chips):
        results.append({
            "pred":        preds_np[j],
            "nodata_mask": chip["nodata_mask"],
            "row_off":     chip["row_off"],
            "col_off":     chip["col_off"],
            "chip_h":      chip["chip_h"],
            "chip_w":      chip["chip_w"],
        })

    if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == n_batches:
        rate = (batch_idx + 1) * BATCH_SIZE / (_time.time() - t_start)
        print(f"  {i1}/{len(chips)} chips  {rate:.1f} chips/s")

preds = results
inference_time = _time.time() - t_start
print(f"[OK] Inference complete — {len(preds)} chips in {inference_time:.1f}s")

# =============================================================================
# PART C — REASSEMBLE MOSAIC
# =============================================================================

scene_h = max(r["row_off"] + r["chip_h"] for r in preds)
scene_w = max(r["col_off"] + r["chip_w"] for r in preds)

vote_accum = np.zeros((NUM_CLASSES, scene_h, scene_w), dtype=np.float32)
weight_map = np.zeros((scene_h, scene_w), dtype=np.float32)

for r in preds:
    r0, c0  = r["row_off"], r["col_off"]
    h,  w   = r["chip_h"],  r["chip_w"]
    valid   = ~r["nodata_mask"][:h, :w]
    pred_hw = r["pred"][:h, :w]
    for cls in range(NUM_CLASSES):
        mask = valid & (pred_hw == cls)
        vote_accum[cls, r0:r0+h, c0:c0+w][mask] += 1.0
    weight_map[r0:r0+h, c0:c0+w][valid] += 1.0

covered    = weight_map > 0
prob_accum = vote_accum.copy()
prob_accum[:, covered] /= weight_map[covered]

scene_pred = np.full((scene_h, scene_w), -1, dtype=np.int16)
scene_pred[covered] = np.argmax(
    prob_accum[:, covered], axis=0
).astype(np.int16)

scene_transform = preds[0].get("transform", chips[0]["transform"])
scene_crs       = preds[0].get("crs",       chips[0]["crs"])

with rasterio.open(
    pred_raster_path, "w",
    driver    = "GTiff",
    height    = scene_pred.shape[0],
    width     = scene_pred.shape[1],
    count     = 1,
    dtype     = np.int16,
    crs       = scene_crs,
    transform = scene_transform,
    nodata    = -1,
) as dst:
    dst.write(scene_pred.astype(np.int16), 1)

print(f"[OK] Scene prediction: {scene_pred.shape}  "
      f"Covered: {covered.sum():,}/{scene_pred.size:,} px")

# =============================================================================
# PART D — CDL VALIDATION
# =============================================================================

sep = "=" * 60
print(sep)
print("  CDL VALIDATION — FINE-TUNED MODEL")
print(sep)

pred_corn_mask = (scene_pred == CORN_CLASS_IDX).astype(np.uint8)
cdl_corn_mask  = (cdl_aligned == CDL_CORN_VALUE).astype(np.uint8)
valid_mask     = (scene_pred >= 0) & (cdl_aligned > 0)

print(f"   Valid pixels for comparison: {valid_mask.sum():,} "
      f"({100 * valid_mask.sum() / valid_mask.size:.1f}% of scene)")

pred_flat = pred_corn_mask[valid_mask]
cdl_flat  = cdl_corn_mask[valid_mask]

TP = int(np.sum((pred_flat == 1) & (cdl_flat == 1)))
FP = int(np.sum((pred_flat == 1) & (cdl_flat == 0)))
FN = int(np.sum((pred_flat == 0) & (cdl_flat == 1)))
TN = int(np.sum((pred_flat == 0) & (cdl_flat == 0)))

precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)
iou       = TP / (TP + FP + FN + 1e-9)
pixel_acc = (TP + TN) / (TP + FP + FN + TN + 1e-9)

ACRES_PER_PIXEL = 900 / 4046.856
pred_corn_acres = int(pred_flat.sum()) * ACRES_PER_PIXEL
cdl_corn_acres  = int(cdl_flat.sum())  * ACRES_PER_PIXEL
acre_diff       = pred_corn_acres - cdl_corn_acres
pct_diff        = acre_diff / (cdl_corn_acres + 1) * 100
sign            = "+" if acre_diff >= 0 else "-"
direction       = "overcount" if acre_diff >= 0 else "undercount"

print()
print(sep)
print("  CORN CLASSIFICATION RESULTS — POLK COUNTY, IOWA (FINE-TUNED)")
print(sep)
print()
print("  Pixel-level metrics (corn vs. non-corn):")
print(f"    True Positives  : {TP:>10,}  px  (corn correctly identified)")
print(f"    False Positives : {FP:>10,}  px  (predicted corn, actually not corn)")
print(f"    False Negatives : {FN:>10,}  px  (corn missed by model)")
print(f"    True Negatives  : {TN:>10,}  px  (non-corn correctly identified)")
print()
print("  Performance:")
print(f"    Precision (corn) : {precision:.3f}")
print(f"    Recall    (corn) : {recall:.3f}")
print(f"    F1 Score  (corn) : {f1:.3f}")
print(f"    IoU       (corn) : {iou:.3f}")
print(f"    Pixel accuracy   : {pixel_acc:.3f}")
print()
print("  Area estimates:")
print(f"    Model predicted corn : {pred_corn_acres:>10,.0f} acres")
print(f"    CDL ground-truth corn: {cdl_corn_acres:>10,.0f} acres")
print(f"    Area difference      : {sign}{abs(acre_diff):>8,.0f} acres "
      f"({sign}{abs(pct_diff):.1f}%  {direction})")
print()

if f1 >= 0.80:
    verdict = "✅  Strong corn detection — field boundaries align well with CDL"
elif f1 >= 0.65:
    verdict = "⚠️   Moderate detection — some confusion with neighbouring crops"
elif f1 >= 0.50:
    verdict = "⚠️   Weak detection — consider additional fine-tuning epochs"
else:
    verdict = "❌  Low detection — check imagery quality / cloud contamination"

print(f"  {verdict}")
print(sep)

# =============================================================================
# PART E — WRITE SUMMARY CSV
# Pulls best_val_f1 and CORN_THRESHOLD from Step 7.5 scope if available.
# =============================================================================

csv_path = OUT_DIR / "iowa_finetune_summary.csv"

# Pull training summary from Step 7.5 checkpoint if available
_ckpt_epoch    = -1
_ckpt_val_f1   = float("nan")
_ckpt_val_loss = float("nan")
if LOCAL_CKPT_PATH.exists():
    try:
        _ckpt = torch.load(LOCAL_CKPT_PATH, map_location="cpu")
        _ckpt_epoch    = int(_ckpt.get("epoch", -1)) + 1   # 1-indexed
        _ckpt_val_f1   = float(_ckpt.get("val_f1",  float("nan")))
        _ckpt_val_loss = float(_ckpt.get("loss",    float("nan")))
    except Exception:
        pass

summary = {
    # Run metadata
    "county":               "Polk County, Iowa",
    "hls_year":             HLS_YEAR,
    "chip_size":            CHIP_SIZE,
    "chip_stride":          CHIP_STRIDE,
    "num_chips":            len(chips),
    "num_timesteps":        NUM_FRAMES,
    # Training
    "finetune_epochs_run":  _ckpt_epoch,
    "finetune_lr":          FINETUNE_LR,
    "finetune_batch":       FINETUNE_BATCH,
    "early_stop_patience":  EARLY_STOP_PATIENCE,
    "best_val_f1_training": round(_ckpt_val_f1,  4),
    "best_val_loss":        round(_ckpt_val_loss, 4),
    # Threshold
    "corn_threshold":       round(globals().get("CORN_THRESHOLD", float("nan")), 2),
    # Validation metrics
    "TP":                   TP,
    "FP":                   FP,
    "FN":                   FN,
    "TN":                   TN,
    "precision":            round(precision, 4),
    "recall":               round(recall,    4),
    "f1":                   round(f1,        4),
    "iou":                  round(iou,       4),
    "pixel_accuracy":       round(pixel_acc, 4),
    # Area
    "pred_corn_acres":      round(pred_corn_acres, 1),
    "cdl_corn_acres":       round(cdl_corn_acres,  1),
    "acre_diff":            round(acre_diff,        1),
    "acre_diff_pct":        round(pct_diff,         2),
    "acre_diff_direction":  direction,
    # Inference
    "inference_time_s":     round(inference_time, 1),
    "scene_h":              scene_h,
    "scene_w":              scene_w,
    "covered_px":           int(covered.sum()),
    "total_px":             int(scene_pred.size),
    # Verdict
    "verdict":              verdict.replace("✅","OK").replace("⚠️","WARN").replace("❌","FAIL"),
}

write_header = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary.keys()))
    if write_header:
        writer.writeheader()
    writer.writerow(summary)

print(f"\n[OK] Summary appended → {csv_path}")
print(f"     f1={f1:.4f}  precision={precision:.4f}  recall={recall:.4f}  "
      f"threshold={globals().get('CORN_THRESHOLD', float('nan')):.2f}")

  STEP 7.6 — RE-RUNNING PIPELINE WITH FINE-TUNED WEIGHTS
  Loading weights from: trained_model/prithvi_crop_finetuned.pt

  LOADING PRITHVI-EO-1.0-100M CROP CLASSIFICATION MODEL
   Using fine-tuned weights : trained_model/prithvi_crop_finetuned.pt
   Missing (non-aux) : 0
   ✅ Model ready (182.9M params)
Running inference on 41 chips (batch=4)...
  20/41 chips  32.7 chips/s
  40/41 chips  45.1 chips/s
  41/41 chips  48.7 chips/s
[OK] Inference complete — 41 chips in 0.9s
[OK] Scene prediction: (1422, 1350)  Covered: 1,702,114/1,919,700 px
  CDL VALIDATION — FINE-TUNED MODEL
   Valid pixels for comparison: 1,701,422 (88.6% of scene)

  CORN CLASSIFICATION RESULTS — POLK COUNTY, IOWA (FINE-TUNED)

  Pixel-level metrics (corn vs. non-corn):
    True Positives  :    278,161  px  (corn correctly identified)
    False Positives :     45,485  px  (predicted corn, actually not corn)
    False Negatives :     63,114  px  (corn missed by model)
    True Negatives  :  1,314,662  px  (non-corn cor

In [13]:
# =============================================================================
# IOWA STATE LOOP — ALL 99 COUNTIES — FINE-TUNED WEIGHTS + VISUALIZATION
# =============================================================================

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import torch
import rasterio
import shutil
import time
import json
import csv
from pathlib import Path
from rasterio.warp import reproject as rio_reproject
from rasterio.enums import Resampling

# =============================================================================
# GUARDS
# =============================================================================

for _var, _step in [
    ("chips",            "Step 4"),
    ("load_crop_model",  "Step 5"),
    ("CROP_MODEL_LOCAL", "Step 7.5"),
    ("HLS_TIMESTEPS",    "Step 1"),
    ("CDL_DIR",          "Step 1"),
    ("HLS_YEAR",         "Step 1"),
    ("NUM_CLASSES",      "Step 1"),
    ("CORN_CLASS_IDX",   "Step 1"),
    ("CDL_CORN_VALUE",   "Step 1"),
    ("CHIP_SIZE",        "Step 1"),
    ("CHIP_STRIDE",      "Step 1"),
    ("NUM_FRAMES",       "Step 1"),
]:
    if _var not in globals():
        raise RuntimeError(f"'{_var}' not found — run {_step} first.")

# =============================================================================
# PART A — LOAD FINE-TUNED MODEL ONCE — ON GPU IF AVAILABLE
# =============================================================================

print("=" * 60)
print("  LOADING FINE-TUNED WEIGHTS")
print("=" * 60)
print(f"  Checkpoint : {CROP_MODEL_LOCAL}")

# FIX — was hardcoded .cpu(); now uses GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = load_crop_model()
model  = model.to(device)
model.eval()

if device.type == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
print(f"[OK] Fine-tuned model loaded → {device}")

# =============================================================================
# SETUP
# =============================================================================

IOWA_DIR = Path("./iowa_predictions")
MAPS_DIR = IOWA_DIR / "maps"
IOWA_DIR.mkdir(exist_ok=True)
MAPS_DIR.mkdir(exist_ok=True)

ACRES_PER_PIXEL = 900 / 4046.856
BATCH_SIZE      = 8 if device.type == "cuda" else 4   # larger batch on GPU

# Load all Iowa counties
print("\nLoading Iowa county boundaries...")
iowa_counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2022/COUNTY/tl_2022_us_county.zip"
).query("STATEFP == '19'").to_crs("EPSG:4326")
iowa_counties = iowa_counties.sort_values("NAME").reset_index(drop=True)
print(f"  {len(iowa_counties)} counties loaded")

CDL_STATE_PATH = CDL_DIR / f"cdl_{HLS_YEAR}_state_19_raw.tif"
if not CDL_STATE_PATH.exists():
    raise FileNotFoundError(
        f"Iowa state CDL not found: {CDL_STATE_PATH}\n"
        "Make sure Step 3 ran successfully."
    )

# Resume support — read completed counties from existing CSV
summary_path = IOWA_DIR / "iowa_finetuned_summary.csv"
results_rows = []
completed    = set()

if summary_path.exists():
    existing     = pd.read_csv(summary_path)
    completed    = set(existing["county"].tolist())
    results_rows = existing.to_dict("records")
    print(f"\n  Resuming — {len(completed)} counties already done")

session        = build_earthdata_session()
total_counties = len(iowa_counties)
t_loop_start   = time.time()


# =============================================================================
# HELPERS
# =============================================================================

def _vote_mosaic(preds_list, scene_h, scene_w, num_classes):
    """
    One-hot vote accumulation across overlapping chips → argmax.
    Replaces the broken label-averaging (accum/weights) from original code.
    """
    vote_accum = np.zeros((num_classes, scene_h, scene_w), dtype=np.float32)
    weight_map = np.zeros((scene_h, scene_w), dtype=np.float32)

    for r in preds_list:
        r0, c0  = r["row_off"], r["col_off"]
        h,  w   = r["chip_h"],  r["chip_w"]
        valid   = ~r["nodata_mask"][:h, :w]
        pred_hw = r["pred"][:h, :w]
        for cls in range(num_classes):
            mask = valid & (pred_hw == cls)
            vote_accum[cls, r0:r0+h, c0:c0+w][mask] += 1.0
        weight_map[r0:r0+h, c0:c0+w][valid] += 1.0

    covered    = weight_map > 0
    prob_accum = vote_accum
    prob_accum[:, covered] /= weight_map[covered]

    scene_pred = np.full((scene_h, scene_w), -1, dtype=np.int16)
    scene_pred[covered] = np.argmax(
        prob_accum[:, covered], axis=0
    ).astype(np.int16)

    return scene_pred, covered


def _run_inference(chip_list, mdl, dev, batch_size):
    """Run model inference on a list of chips. Returns list of pred dicts."""
    results   = []
    n_batches = (len(chip_list) + batch_size - 1) // batch_size
    t0        = time.time()

    for b_idx in range(n_batches):
        i0 = b_idx * batch_size
        i1 = min(i0 + batch_size, len(chip_list))
        bc = chip_list[i0:i1]
        x  = np.stack([c["tensor"] for c in bc], axis=0)
        with torch.no_grad():
            logits   = mdl(torch.from_numpy(x).float().to(dev))
        preds_np = torch.argmax(logits, dim=1).cpu().numpy()
        for j, chip in enumerate(bc):
            results.append({
                "pred":        preds_np[j],
                "nodata_mask": chip["nodata_mask"],
                "row_off":     chip["row_off"],
                "col_off":     chip["col_off"],
                "chip_h":      chip["chip_h"],
                "chip_w":      chip["chip_w"],
            })

        if (b_idx + 1) % 10 == 0 or (b_idx + 1) == n_batches:
            rate = (b_idx + 1) * batch_size / (time.time() - t0)
            print(f"    {i1}/{len(chip_list)} chips  {rate:.1f} chips/s")

    return results


def plot_county_validation_map(
    county_name, scene_pred, cdl_aligned_arr, hls_stack_path,
    TP, FP, FN, TN, f1, out_path
):
    """4-panel validation map: HLS RGB | Model | CDL | Agreement."""
    try:
        with rasterio.open(hls_stack_path) as src:
            rgb = src.read([3, 2, 1]).astype(np.float32)
        for b in range(3):
            valid_px = rgb[b][rgb[b] > 0]
            if valid_px.size:
                p2, p98 = np.percentile(valid_px, (2, 98))
                rgb[b]  = np.clip((rgb[b] - p2) / (p98 - p2 + 1e-6), 0, 1)
        rgb      = np.moveaxis(rgb, 0, -1)
        have_rgb = True
    except Exception:
        have_rgb = False

    h = scene_pred.shape[0]
    w = scene_pred.shape[1]
    if have_rgb:
        h = min(h, rgb.shape[0])
        w = min(w, rgb.shape[1])

    pred_crop = scene_pred[:h, :w]
    cdl_crop  = cdl_aligned_arr[:h, :w]
    nodata    = pred_crop < 0

    WHITE = [1.0, 1.0, 1.0]
    DARK  = [0.12, 0.12, 0.12]

    pred_img = np.full((h, w, 3), DARK, dtype=np.float32)
    pred_img[pred_crop == CORN_CLASS_IDX] = [1.0, 0.85, 0.0]
    pred_img[nodata] = WHITE

    cdl_img = np.full((h, w, 3), DARK, dtype=np.float32)
    cdl_img[cdl_crop == CDL_CORN_VALUE] = [0.2, 0.7, 0.2]
    cdl_img[nodata] = WHITE

    p_corn    = (pred_crop == CORN_CLASS_IDX)
    c_corn    = (cdl_crop  == CDL_CORN_VALUE)
    agreement = np.zeros((h, w), dtype=np.uint8)
    agreement[ p_corn &  c_corn & ~nodata] = 3
    agreement[ p_corn & ~c_corn & ~nodata] = 2
    agreement[~p_corn &  c_corn & ~nodata] = 1

    agree_colors = np.array([
        DARK, [1.0, 0.5, 0.0], [0.9, 0.1, 0.1], [0.1, 0.8, 0.2],
    ], dtype=np.float32)
    agree_img          = agree_colors[agreement]
    agree_img[nodata]  = WHITE

    n_panels  = 4 if have_rgb else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
    fig.patch.set_facecolor("#1a1a2e")
    title_color = "white"
    fig.suptitle(
        f"{county_name} County, Iowa — Corn Classification ({HLS_YEAR})",
        fontsize=13, fontweight="bold", color=title_color, y=1.01,
    )

    panel_idx = 0
    if have_rgb:
        rgb_crop = rgb[:h, :w].copy()
        rgb_crop[nodata] = WHITE
        axes[panel_idx].imshow(rgb_crop)
        axes[panel_idx].set_title("HLS RGB (T2 mid-season)", color=title_color, fontsize=10)
        axes[panel_idx].axis("off")
        panel_idx += 1

    axes[panel_idx].imshow(pred_img)
    axes[panel_idx].set_title("Model  (gold = corn)", color=title_color, fontsize=10)
    axes[panel_idx].axis("off")
    panel_idx += 1

    axes[panel_idx].imshow(cdl_img)
    axes[panel_idx].set_title("CDL ground truth  (green = corn)", color=title_color, fontsize=10)
    axes[panel_idx].axis("off")
    panel_idx += 1

    axes[panel_idx].imshow(agree_img)
    axes[panel_idx].set_title(f"Agreement  F1={f1:.3f}", color=title_color, fontsize=10)
    axes[panel_idx].axis("off")
    axes[panel_idx].legend(
        handles=[
            mpatches.Patch(color=[0.1, 0.8, 0.2], label=f"TP  {TP:,}"),
            mpatches.Patch(color=[0.9, 0.1, 0.1], label=f"FP  {FP:,}"),
            mpatches.Patch(color=[1.0, 0.5, 0.0], label=f"FN  {FN:,}"),
            mpatches.Patch(color=[0.12, 0.12, 0.12], label="TN"),
            mpatches.Patch(color=[1.0, 1.0, 1.0], label="Nodata"),
        ],
        loc="lower right", fontsize=7.5, framealpha=0.85,
    )
    for ax in axes:
        for spine in ax.spines.values():
            spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"   Map saved → {out_path.name}")


# =============================================================================
# COUNTY LOOP
# =============================================================================

for idx, row in iowa_counties.iterrows():
    county_name = row["NAME"]
    county_geom = row["geometry"]

    if county_name in completed:
        print(f"  [{idx+1:>2}/{total_counties}] {county_name:<20} ✓ cached")
        continue

    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  [{idx+1:>2}/{total_counties}] {county_name} County, Iowa")
    print(sep)
    t_county = time.time()

    bounds      = county_geom.bounds
    county_bbox = {
        "min_lon": bounds[0], "min_lat": bounds[1],
        "max_lon": bounds[2], "max_lat": bounds[3],
    }

    county_hls_dir   = IOWA_DIR / "temp_hls" / county_name.replace(" ", "_")
    county_pred_path = IOWA_DIR / f"{county_name.replace(' ', '_')}_pred_finetuned.tif"
    county_hls_dir.mkdir(parents=True, exist_ok=True)

    try:
        # ── DOWNLOAD HLS ──────────────────────────────────────────────────
        county_stacks = []
        download_ok   = True

        for t_idx, date_range in enumerate(HLS_TIMESTEPS):
            band_paths = download_all_granules_for_timestep(
                t_idx, date_range, county_bbox, session
            )
            if band_paths is None:
                print(f"  ⚠️  No granules for T{t_idx+1} — skipping county")
                download_ok = False
                break

            stack_path = county_hls_dir / f"{county_name}_t{t_idx}_stacked.tif"
            mosaic_and_clip_bands(band_paths, county_geom, stack_path)
            county_stacks.append(stack_path)

            # Delete raw band tiles immediately to save disk
            for paths in band_paths.values():
                for p in paths:
                    if p.exists() and "stacked" not in p.name:
                        p.unlink(missing_ok=True)

        if not download_ok or len(county_stacks) != NUM_FRAMES:
            print(f"  ❌ Skipping {county_name} — incomplete downloads")
            results_rows.append({
                "county": county_name, "status": "skipped",
                "corn_px": 0, "corn_acres": 0, "cdl_acres": 0,
                "area_diff_pct": None,
                "f1": None, "precision": None, "recall": None,
                "iou": None, "pixel_acc": None,
                "TP": 0, "FP": 0, "FN": 0, "TN": 0,
                "elapsed_s": round(time.time() - t_county, 1),
            })
            shutil.rmtree(county_hls_dir, ignore_errors=True)
            pd.DataFrame(results_rows).to_csv(summary_path, index=False)
            continue

        # ── CHIP ──────────────────────────────────────────────────────────
        county_chips = extract_chips(county_stacks)
        print(f"  Chips: {len(county_chips)}")

        # ── INFERENCE — GPU-accelerated ───────────────────────────────────
        print(f"  Running inference ({device})...")
        county_preds = _run_inference(county_chips, model, device, BATCH_SIZE)
        print(f"  [OK] {len(county_preds)} chip predictions")

        # ── MOSAIC — vote accumulation (not label averaging) ──────────────
        scene_h = max(r["row_off"] + r["chip_h"] for r in county_preds)
        scene_w = max(r["col_off"] + r["chip_w"] for r in county_preds)
        county_pred, covered = _vote_mosaic(county_preds, scene_h, scene_w, NUM_CLASSES)

        county_transform = county_chips[0]["transform"]
        county_crs       = county_chips[0]["crs"]

        with rasterio.open(
            county_pred_path, "w", driver="GTiff",
            height=county_pred.shape[0], width=county_pred.shape[1],
            count=1, dtype=np.int16,
            crs=county_crs, transform=county_transform, nodata=-1,
        ) as dst:
            dst.write(county_pred.astype(np.int16), 1)

        print(f"  [OK] Mosaic: {county_pred.shape}  "
              f"Covered: {covered.sum():,}/{county_pred.size:,} px")

        # ── CDL VALIDATION ────────────────────────────────────────────────
        with rasterio.open(CDL_STATE_PATH) as cdl_ds:
            cdl_county = np.zeros((scene_h, scene_w), dtype=np.uint8)
            rio_reproject(
                source        = rasterio.band(cdl_ds, 1),
                destination   = cdl_county,
                src_transform = cdl_ds.transform,
                src_crs       = cdl_ds.crs,
                dst_transform = county_transform,
                dst_crs       = county_crs,
                resampling    = Resampling.nearest,
            )

        pred_corn_mask = (county_pred == CORN_CLASS_IDX).astype(np.uint8)
        cdl_corn_mask  = (cdl_county  == CDL_CORN_VALUE).astype(np.uint8)
        valid_mask     = (county_pred >= 0) & (cdl_county > 0)

        pred_flat = pred_corn_mask[valid_mask]
        cdl_flat  = cdl_corn_mask[valid_mask]

        TP = int(np.sum((pred_flat == 1) & (cdl_flat == 1)))
        FP = int(np.sum((pred_flat == 1) & (cdl_flat == 0)))
        FN = int(np.sum((pred_flat == 0) & (cdl_flat == 1)))
        TN = int(np.sum((pred_flat == 0) & (cdl_flat == 0)))

        precision = TP / (TP + FP + 1e-9)
        recall    = TP / (TP + FN + 1e-9)
        f1        = 2 * precision * recall / (precision + recall + 1e-9)
        iou       = TP / (TP + FP + FN + 1e-9)
        pixel_acc = (TP + TN) / (TP + FP + FN + TN + 1e-9)

        corn_px         = int((county_pred == CORN_CLASS_IDX).sum())
        pred_corn_acres = corn_px * ACRES_PER_PIXEL
        cdl_corn_acres  = int(cdl_flat.sum()) * ACRES_PER_PIXEL
        acre_diff       = pred_corn_acres - cdl_corn_acres
        area_diff_pct   = acre_diff / (cdl_corn_acres + 1) * 100
        sign            = "+" if acre_diff >= 0 else "-"
        direction       = "overcount" if acre_diff >= 0 else "undercount"

        print(f"\n  F1={f1:.3f}  P={precision:.3f}  R={recall:.3f}  IoU={iou:.3f}")
        print(f"  Corn: model={pred_corn_acres:,.0f} ac  CDL={cdl_corn_acres:,.0f} ac  "
              f"diff={sign}{abs(acre_diff):,.0f} ac ({sign}{abs(area_diff_pct):.1f}%  {direction})")
        print(f"  TP={TP:,}  FP={FP:,}  FN={FN:,}  TN={TN:,}")

        if f1 >= 0.80:
            print("  ✅ Strong detection")
        elif f1 >= 0.65:
            print("  ⚠️  Moderate detection")
        elif f1 >= 0.50:
            print("  ⚠️  Weak detection")
        else:
            print("  ❌ Low detection")

        # ── PER-COUNTY MAP ────────────────────────────────────────────────
        map_out        = MAPS_DIR / f"{county_name.replace(' ', '_')}_corn_validation_map.png"
        hls_rgb_source = county_stacks[1] if len(county_stacks) > 1 else county_stacks[0]
        plot_county_validation_map(
            county_name    = county_name,
            scene_pred     = county_pred,
            cdl_aligned_arr = cdl_county,
            hls_stack_path = hls_rgb_source,
            TP=TP, FP=FP, FN=FN, TN=TN, f1=f1,
            out_path       = map_out,
        )

        elapsed = time.time() - t_county
        print(f"  County wall-clock: {elapsed:.0f}s")

        results_rows.append({
            "county":        county_name,
            "status":        "ok",
            "corn_px":       corn_px,
            "corn_acres":    round(pred_corn_acres, 1),
            "cdl_acres":     round(cdl_corn_acres,  1),
            "area_diff_pct": round(area_diff_pct,   2),
            "f1":            round(f1,        4),
            "precision":     round(precision, 4),
            "recall":        round(recall,    4),
            "iou":           round(iou,       4),
            "pixel_acc":     round(pixel_acc, 4),
            "TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "elapsed_s":     round(elapsed, 1),
        })
        completed.add(county_name)

    except Exception as e:
        print(f"  ❌ Error — {county_name}: {e}")
        results_rows.append({
            "county": county_name, "status": "error",
            "corn_px": 0, "corn_acres": 0, "cdl_acres": 0,
            "area_diff_pct": None,
            "f1": None, "precision": None, "recall": None,
            "iou": None, "pixel_acc": None,
            "TP": 0, "FP": 0, "FN": 0, "TN": 0,
            "elapsed_s": round(time.time() - t_county, 1),
        })

    finally:
        # Always clean up temp HLS tiles
        shutil.rmtree(county_hls_dir, ignore_errors=True)

    # Flush CSV after every county so resume always has fresh data
    pd.DataFrame(results_rows).to_csv(summary_path, index=False)

    counties_done = len(completed)
    elapsed_total = time.time() - t_loop_start
    rate          = counties_done / elapsed_total if elapsed_total > 0 else 1
    eta_s         = (total_counties - counties_done) / rate if rate > 0 else 0
    print(f"  Progress: {counties_done}/{total_counties}  |  ETA: {eta_s/3600:.1f} hrs")


# =============================================================================
# STATEWIDE SUMMARY — TEXT
# =============================================================================

print("\n" + "=" * 60)
print("  IOWA STATEWIDE SUMMARY — FINE-TUNED MODEL")
print("=" * 60)

df = pd.DataFrame(results_rows)
df.to_csv(summary_path, index=False)
ok = df[df["status"] == "ok"].copy()

print(f"\n  Counties processed  : {len(ok)} / {total_counties}")
print(f"  Counties skipped    : {(df['status'] == 'skipped').sum()}")
print(f"  Counties errored    : {(df['status'] == 'error').sum()}")
print(f"  Total corn (model)  : {ok['corn_acres'].sum():>10,.0f} acres")
print(f"  Total corn (CDL)    : {ok['cdl_acres'].sum():>10,.0f} acres")
print(f"  Mean F1             : {ok['f1'].mean():.3f}")
print(f"  Median F1           : {ok['f1'].median():.3f}")
print(f"  Min F1              : {ok['f1'].min():.3f}  ({ok.loc[ok['f1'].idxmin(), 'county']})")
print(f"  Max F1              : {ok['f1'].max():.3f}  ({ok.loc[ok['f1'].idxmax(), 'county']})")
print(f"  Counties F1 ≥ 0.80  : {(ok['f1'] >= 0.80).sum()}")
print(f"  Counties F1 ≥ 0.65  : {(ok['f1'] >= 0.65).sum()}")
print(f"  Counties F1 ≥ 0.50  : {(ok['f1'] >= 0.50).sum()}")
print(f"  Counties F1 < 0.50  : {(ok['f1'] < 0.50).sum()}")

statewide_json = {
    "year":  HLS_YEAR,
    "model": "Prithvi-EO-1.0-100M-multi-temporal-crop-classification (fine-tuned)",
    "counties_ok":              int(len(ok)),
    "counties_skipped":         int((df["status"] == "skipped").sum()),
    "counties_error":           int((df["status"] == "error").sum()),
    "total_corn_model_acres":   round(float(ok["corn_acres"].sum()), 1),
    "total_corn_cdl_acres":     round(float(ok["cdl_acres"].sum()),  1),
    "mean_f1":                  round(float(ok["f1"].mean()),   4),
    "median_f1":                round(float(ok["f1"].median()), 4),
    "min_f1":                   round(float(ok["f1"].min()),    4),
    "max_f1":                   round(float(ok["f1"].max()),    4),
}
json_path = IOWA_DIR / "iowa_statewide_summary.json"
with open(json_path, "w") as fh:
    json.dump(statewide_json, fh, indent=2)
print(f"\n  Summary JSON saved : {json_path}")


# =============================================================================
# STATEWIDE VISUALIZATION — 3 FIGURES
# =============================================================================

print("\n" + "=" * 60)
print("  GENERATING STATEWIDE SUMMARY PLOTS")
print("=" * 60)

DARK_BG  = "#1a1a2e"
TEXT_COL = "white"
ACCENT   = "#e8b84b"
ACCENT2  = "#4bc8e8"
GRID_COL = "#333355"

plt.rcParams.update({
    "figure.facecolor": DARK_BG,
    "axes.facecolor":   DARK_BG,
    "axes.edgecolor":   GRID_COL,
    "axes.labelcolor":  TEXT_COL,
    "xtick.color":      TEXT_COL,
    "ytick.color":      TEXT_COL,
    "text.color":       TEXT_COL,
    "grid.color":       GRID_COL,
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
})

# ── Figure 1 — F1 choropleth ───────────────────────────────────────────────
print("\n  [1/3] F1 choropleth map...")

county_f1 = iowa_counties.merge(
    ok[["county", "f1", "corn_acres", "cdl_acres"]],
    left_on="NAME", right_on="county", how="left"
)

fig, ax = plt.subplots(1, 1, figsize=(14, 8), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

cmap = plt.cm.RdYlGn
county_f1.dropna(subset=["f1"]).plot(
    column="f1", ax=ax, cmap=cmap, vmin=0.0, vmax=1.0,
    edgecolor="#444466", linewidth=0.4,
)
county_f1_miss = county_f1[county_f1["f1"].isna()]
if not county_f1_miss.empty:
    county_f1_miss.plot(ax=ax, color="#333344", edgecolor="#555577", linewidth=0.4)

label_rows = pd.concat([
    county_f1.dropna(subset=["f1"]).nlargest(5, "f1"),
    county_f1.dropna(subset=["f1"]).nsmallest(5, "f1"),
]).drop_duplicates(subset="NAME")

for _, r in label_rows.iterrows():
    cx, cy = r.geometry.centroid.x, r.geometry.centroid.y
    ax.annotate(
        r["NAME"], xy=(cx, cy), fontsize=6, ha="center", va="center",
        color="white", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.15", fc="#00000066", ec="none"),
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("F1 Score (corn vs. non-corn)", fontsize=10)
cbar.ax.yaxis.set_tick_params(color=TEXT_COL)

ax.set_title(
    f"Iowa Corn Classification F1 — Fine-Tuned Model ({HLS_YEAR})\n"
    f"Mean F1 = {ok['f1'].mean():.3f}   |   "
    f"Counties ≥ 0.65: {(ok['f1'] >= 0.65).sum()}  |  "
    f"< 0.50: {(ok['f1'] < 0.50).sum()}",
    fontsize=12, fontweight="bold", color=TEXT_COL, pad=12,
)
ax.set_axis_off()
plt.tight_layout()
choro_path = IOWA_DIR / "iowa_f1_choropleth.png"
plt.savefig(choro_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.close()
print(f"   Saved → {choro_path.name}")

# ── Figure 2 — Scatter: Model acres vs CDL acres ──────────────────────────
print("  [2/3] Model vs CDL acres scatter...")

fig, ax = plt.subplots(figsize=(9, 8), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

norm      = mcolors.Normalize(vmin=0, vmax=1)
scatter_c = cm.RdYlGn(norm(ok["f1"].values))
ax.scatter(
    ok["cdl_acres"], ok["corn_acres"],
    c=scatter_c, s=60, edgecolors="#ffffff44", linewidths=0.5, zorder=3,
)
max_val = max(ok["cdl_acres"].max(), ok["corn_acres"].max()) * 1.05
ax.plot([0, max_val], [0, max_val], "--", color="#ffffff55", linewidth=1.2, label="1:1 line")

for _, r in ok[ok["area_diff_pct"].abs() > 40].iterrows():
    ax.annotate(
        r["county"], xy=(r["cdl_acres"], r["corn_acres"]),
        xytext=(8, 4), textcoords="offset points",
        fontsize=7, color=TEXT_COL, alpha=0.85,
    )

sm2 = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
sm2.set_array([])
cbar2 = fig.colorbar(sm2, ax=ax, fraction=0.025, pad=0.02)
cbar2.set_label("F1 Score", fontsize=10)

ax.set_xlabel("CDL Ground-Truth Corn (acres)", fontsize=11)
ax.set_ylabel("Model Predicted Corn (acres)", fontsize=11)
ax.set_title(
    f"Model vs CDL Corn Area — Iowa Counties ({HLS_YEAR})\n"
    f"Total model: {ok['corn_acres'].sum():,.0f} ac  |  "
    f"Total CDL: {ok['cdl_acres'].sum():,.0f} ac",
    fontsize=12, fontweight="bold", color=TEXT_COL, pad=10,
)
ax.grid(True)
ax.legend(facecolor=DARK_BG, edgecolor=GRID_COL, fontsize=9)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
scatter_path = IOWA_DIR / "iowa_corn_acres_scatter.png"
plt.savefig(scatter_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.close()
print(f"   Saved → {scatter_path.name}")

# ── Figure 3 — Metric distributions ──────────────────────────────────────
print("  [3/3] Metric distribution histograms...")

metrics = [
    ("f1",        "F1 Score",  ACCENT),
    ("precision", "Precision", ACCENT2),
    ("recall",    "Recall",    "#b084e8"),
    ("iou",       "IoU",       "#e87884"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=DARK_BG)
axes = axes.flatten()

for ax, (col, label, color) in zip(axes, metrics):
    vals = ok[col].dropna().values
    ax.hist(vals, bins=20, range=(0, 1), color=color,
            edgecolor=DARK_BG, linewidth=0.6, alpha=0.85)
    ax.axvline(vals.mean(),     color="white",  linestyle="--", linewidth=1.2,
               label=f"Mean  {vals.mean():.3f}")
    ax.axvline(np.median(vals), color="#aaaaaa", linestyle=":",  linewidth=1.2,
               label=f"Median {np.median(vals):.3f}")
    ax.set_title(label, fontsize=11, color=TEXT_COL, fontweight="bold")
    ax.set_xlabel("Score", fontsize=9)
    ax.set_ylabel("Counties", fontsize=9)
    ax.set_xlim(0, 1)
    ax.grid(True, axis="y")
    ax.legend(fontsize=8, facecolor=DARK_BG, edgecolor=GRID_COL)

fig.suptitle(
    f"Metric Distributions Across Iowa Counties — Fine-Tuned Model ({HLS_YEAR})",
    fontsize=13, fontweight="bold", color=TEXT_COL, y=1.01,
)
plt.tight_layout()
dist_path = IOWA_DIR / "iowa_metrics_distributions.png"
plt.savefig(dist_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.close()
print(f"   Saved → {dist_path.name}")


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 60)
print("  ALL DONE")
print("=" * 60)
print(f"\n✅ Per-county maps  : {MAPS_DIR}/  ({len(list(MAPS_DIR.glob('*.png')))} PNGs)")
print(f"✅ F1 choropleth    : {choro_path}")
print(f"✅ Acres scatter    : {scatter_path}")
print(f"✅ Metric histograms: {dist_path}")
print(f"✅ Summary CSV      : {summary_path}")
print(f"✅ Summary JSON     : {json_path}")
print(f"✅ County GeoTIFFs  : {IOWA_DIR}/")
print("\n🌽 Iowa fine-tuned run + visualization complete!")

  LOADING FINE-TUNED WEIGHTS
  Checkpoint : trained_model/prithvi_crop_finetuned.pt
  LOADING PRITHVI-EO-1.0-100M CROP CLASSIFICATION MODEL
   Using fine-tuned weights : trained_model/prithvi_crop_finetuned.pt
   Missing (non-aux) : 0
   ✅ Model ready (182.9M params)
  GPU : NVIDIA A10G
[OK] Fine-tuned model loaded → cuda

Loading Iowa county boundaries...
  99 counties loaded

  [ 1/99] Adair County, Iowa

📡 Timestep 1 — 2022-04-15/2022-04-30
   Found 4 granule(s) for 2022-04-15/2022-04-30
   MGRS tiles: ['T15TUG', 'T15TUF']
   Granule: HLS.S30.T15TUG.2022116T165839.v2.0  (cloud: 0%)
      BLUE... ✓
      GREEN... ✓
      RED... ✓
      NIR_NARROW... ✓
      SWIR_1... ✓
      SWIR_2... ✓
   Granule: HLS.S30.T15TUF.2022116T165839.v2.0  (cloud: 0%)
      BLUE... ✓
      GREEN... ✓
      RED... ✓
      NIR_NARROW... ✓
      SWIR_1... ✓
      SWIR_2... ✓

📡 Timestep 2 — 2022-06-15/2022-06-30
   Found 6 granule(s) for 2022-06-15/2022-06-30
   MGRS tiles: ['T15TUG', 'T15TUF']
   Granule: HL

In [ ]:
# =============================================================================
# STEP 6 — REASSEMBLE CHIP PREDICTIONS INTO FULL-SCENE MOSAIC
# =============================================================================
# Stitches per-chip predictions back into a full county-scale raster.
# Uses weighted-average blending for overlapping chips (stride < chip_size)
# to eliminate seam artifacts at chip boundaries.
#
# Key fixes vs original:
#   - Accepts preds as list of dicts (Step 5 new format) not list of arrays
#   - nodata_mask from Steps 4/5 excluded from blending — pixels outside
#     the county polygon stay -1 (nodata) in the final mosaic
#   - Fixed broken rasterio.open hyperlink syntax error
#   - Prediction raster saved to OUT_DIR not cwd for cleaner output layout
# =============================================================================

from pathlib import Path


def reassemble_predictions(
    pred_results: list[dict],
    chip_size:    int = CHIP_SIZE,
    stride:       int = CHIP_STRIDE,
) -> tuple[np.ndarray, object, object]:
    """
    Stitch per-chip prediction dicts into a full-scene prediction map.

    Args:
        pred_results : list of dicts from Step 5, each containing:
                         'pred'        [H, W] int   — argmax class index
                         'nodata_mask' [H, W] bool  — True = outside county
                         'row_off', 'col_off'        — chip position in scene
                         'chip_h', 'chip_w'          — valid (unpadded) extent
        chip_size    : chip side length in pixels (224)
        stride       : sliding window stride (196)

    Returns:
        scene_pred  : np.ndarray [H, W] int16
                      predicted class index per pixel;
                      -1 where no valid chip contributed (nodata / outside county)
        transform   : rasterio.Affine  (from chip metadata)
        crs         : rasterio.CRS    (from chip metadata)
    """
    if not pred_results:
        raise ValueError("pred_results is empty — did inference run correctly?")

    # Recover scene dimensions from the last chip's position + extent
    # (chips cover the full scene so max row/col + extent = scene size)
    scene_h = max(r["row_off"] + r["chip_h"] for r in pred_results)
    scene_w = max(r["col_off"] + r["chip_w"] for r in pred_results)

    # Accumulator: sum of class votes; weight map: number of chips per pixel
    accum   = np.zeros((scene_h, scene_w), dtype=np.float32)
    weights = np.zeros((scene_h, scene_w), dtype=np.float32)

    for r in pred_results:
        r0, c0 = r["row_off"], r["col_off"]
        h,  w  = r["chip_h"],  r["chip_w"]

        pred_crop   = r["pred"][:h, :w].astype(np.float32)      # [h, w]
        nodata_crop = r["nodata_mask"][:h, :w]                  # [h, w] bool

        # Only accumulate valid (inside county) pixels
        valid = ~nodata_crop
        accum  [r0:r0+h, c0:c0+w][valid] += pred_crop[valid]
        weights[r0:r0+h, c0:c0+w][valid] += 1.0

    # Build final prediction map
    # Pixels with no valid chip contribution stay -1 (nodata)
    scene_pred = np.full((scene_h, scene_w), fill_value=-1, dtype=np.int16)
    covered    = weights > 0
    scene_pred[covered] = np.round(
        accum[covered] / weights[covered]
    ).astype(np.int16)

    # Pull geo metadata from any result (all chips share the same scene grid)
    transform = pred_results[0].get("transform", None)
    crs       = pred_results[0].get("crs",       None)

    # If transform/crs not in pred_results, pull from chips list
    # (chips list is still in scope from Step 4)
    if transform is None:
        transform = chips[0]["transform"]
    if crs is None:
        crs = chips[0]["crs"]

    return scene_pred, transform, crs


# =============================================================================
# RUN — REASSEMBLE
# =============================================================================
sep = "=" * 60
print(sep)
print("  REASSEMBLING CHIP PREDICTIONS INTO FULL-SCENE MOSAIC")
print(sep)

scene_pred, scene_transform, scene_crs = reassemble_predictions(preds)

covered_px = int(np.sum(scene_pred >= 0))
total_px   = scene_pred.size
unique, counts = np.unique(scene_pred[scene_pred >= 0], return_counts=True)

print(f"[OK]  Scene prediction shape : {scene_pred.shape}  (H, W)")
print(f"[OK]  Covered pixels         : {covered_px:,} / {total_px:,} "
      f"({100 * covered_px / total_px:.1f}%)")
print()
print("      Class distribution (valid pixels):")
for cls, cnt in zip(unique, counts):
    name  = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"class_{cls}"
    acres = cnt * (900 / 4046.856)
    print(f"        {cls:>2}  {name:<22} : {cnt:>10,} px  (~{acres:>10,.0f} acres)")

# =============================================================================
# WRITE PREDICTION RASTER TO DISK  (required by Step 7)
# =============================================================================
pred_raster_path = PRED_DIR / "scene_pred.tif"

with rasterio.open(
    pred_raster_path,
    mode      = "w",
    driver    = "GTiff",
    height    = scene_pred.shape[0],
    width     = scene_pred.shape[1],
    count     = 1,
    dtype     = np.int16,
    crs       = scene_crs,
    transform = scene_transform,
    nodata    = -1,
) as dst:
    dst.write(scene_pred.astype(np.int16), 1)

print()
print(f"[OK]  Prediction raster written : {pred_raster_path}")
print(f"      Shape     : {scene_pred.shape}  (H, W)")
print(f"      CRS       : {scene_crs}")
print(f"      Nodata    : -1")
print(sep)